<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Bayesian-Embeddings/blob/main/Sim2_bayesian_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# SIMULATION 2 — ONE SELF-CONTAINED GOOGLE COLAB CELL
# Fixed-maximum-rank multiplicative spectral shrinkage: NO RJMCMC and NO discrete rank sampler.
# V6 exact-tail mixing revision: multiscale prior-corrected Beta MH + logit random walk, wider spectral slice,
# dense parallel tempering, blocked Gaussianized-Beta updates, cached/JIT likelihood.
# Hierarchical Beta spectral ratios and predictor-separable risk-sensitive allocation.
# Convergence is measured, not assumed. Exact beta-tempered, fixed-dimension posterior sweeps; minimum ESS gate unchanged.
# Checkpoints saved to Google Drive every 20 minutes and at phase boundaries.
# The new MCMC update schedule starts fresh chains. Optional V5/V4 import reuses data/reference fit ONLY;
# V6 checkpoints resume exactly. All old checkpoints are read-only.
# Only MAIN RESULTS and RECOVERY checkpoints go to disk; ALL MCMC DIAGNOSTICS PRINT TO CONSOLE.
# ===========================================================================
# SIMULATION 2: INDEPENDENT CODE AND CHECKPOINTS; no other notebook cells required.
# EDIT THESE BEFORE RUNNING A NEW EXPERIMENT:
SIMULATIONS = ('2',)      # Fixed for Simulation 2; independently runnable
RUN_FULL = False         # False: 1 replication per regime (diagnostic); True: paper-scale repetitions
RISK_XI = (0.0, 0.5, 1.0)  # fixed, predeclared uncertainty penalties
REQUIRE_CONVERGENCE = True   # do not interpret/refit failed Bayesian chains
STOP_ON_FIRST_FAILURE = True # stop expensive replications so diagnostics can be inspected
# MCMC settings apply PER independent chain. Extensions are automatic until pass or limit.
N_CHAINS = 4
MCMC_BASE = 4000 if not RUN_FULL else 5000
MCMC_EXTEND = 2000
MCMC_MAX = 12000 if not RUN_FULL else 20000
MCMC_BURN = 2000 if not RUN_FULL else 3000
MCMC_THIN = 2 if not RUN_FULL else 4
RHAT_LIMIT = 1.01          # strict R-hat gate for ALL reported posterior parameters
ESS_BULK_MIN = 400
ESS_TAIL_MIN = 400
CHECKPOINT_SECONDS = 20 * 60
# Temperatures temper ONLY the collapsed likelihood; priors stay untempered.
PT_TEMPS = 20 if not RUN_FULL else 24
PT_MIN_BETA = 0.20 if not RUN_FULL else 0.12
PT_SWAP_EVERY = 2
PT_ADAPT_EVERY = 100   # temperature-ladder adaptation stops at burn-in
PT_TARGET_SWAP = 0.25
PT_MIN_SWAP_WARN = 0.08
PT_GLOBAL_MOVE_EVERY = 8  # occasional joint orientation and regression move
PT_PRIOR_REFRESH_EVERY = 10  # retain original exact independent conditional-prior ratio proposals
PT_HOT_SWEEP_EVERY = 2       # each hot replica updates every second sweep; cold updates every sweep
PT_HOT_PREDICTORS = 4         # cyclic subset of the 12 predictors per hot sweep; conditional kernels preserve tempered target
PT_SPECTRAL_BLOCK_EVERY = 6  # exact Gaussianized-Beta joint ratio ESS proposal (cold chain)
PT_COUPLED_TAIL_EVERY = 4    # exact joint weak-tail orientation and ratio ESS proposal (cold chain)
# Targeted, exact MH moves for near-zero trailing spectral ratios.
# Indices below are ZERO-BASED ratio positions, not eigenvalue positions.
# The selection schedule is fixed ahead of sampling (not state-dependent).
TAIL_PRIOR_MH_EVERY = 2
TAIL_PRIOR_MH_CUTOFFS = (0.02, 1e-4, 1e-8)  # coarse/medium/extremely small ratios
TAIL_PRIOR_MH_WEIGHTS = (0.22, 0.26, 0.27, 0.25)  # first weight: unrestricted Beta
TAIL_PRIOR_MH_INDEP_PROB = 0.55  # else a symmetric logit-scale random walk
TAIL_PRIOR_MH_RW_SCALES = (2.0, 7.0)  # fixed predeclared scales; no warmup adaptation
TAIL_RATIO_SLICE_WIDTH = 2.5   # broad logit-slice intervals; still exact
TAIL_FOCUS = {2: (1,2), 3: (0,1), 5: (1,2), 7: (0,1,2,3),
              8: (1,), 10: (0,), 11: (2,3)}
# Extra attention to j=8 eta_1 and j=12 eta_3 (the two worst diagnostic areas).
TAIL_PRIORITY = ((7,0), (11,2))
# All extra moves are valid invariant kernels. Hot subset steps reduce cost of the denser ladder;
# their mixing efficiency must still be verified, not assumed.
# Do not misinterpret a failed posterior as 'success' based on high median ESS.
# All parameters are reported; strict convergence gate is applied to all.

# ===========================================================================
import os, time, json, pickle, random, hashlib, gc, math, warnings, shutil, traceback
from datetime import datetime
import numpy as np
import pandas as pd
from scipy.special import expit, log_expit, gammaln, betaln, ndtr, ndtri, betainc, betaincinv
import torch
from torch import nn
from torch.nn import functional as F
try:
    import arviz as az
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'arviz'])
    import arviz as az
warnings.filterwarnings('ignore', category=FutureWarning)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(max(1,min(4,os.cpu_count() or 1)))
# Tiny (5x5) NumPy/BLAS operations dominate the MCMC inner loop; avoid
# expensive multithread startup on every candidate. This changes no target.
try:
    from threadpoolctl import threadpool_limits
    _blas_guard = threadpool_limits(limits=1,user_api='blas')
    print('NumPy BLAS threads limited to 1 for small-matrix MCMC computations.',flush=True)
except ImportError:
    _blas_guard = None
print('Torch',torch.__version__,'| device:',DEVICE,'| sampler: cached likelihood/auto JIT + conditional hyperprior; neural nets: torch GPU if available',flush=True)
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as exc:
        raise RuntimeError('Google Drive mount failed; authorize access and rerun. Drive checkpoint is mandatory in Colab.') from exc
    ROOT = '/content/drive/MyDrive/JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V6_TAIL_MH'
except ImportError:
    ROOT = os.path.abspath('./JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V6_TAIL_MH')
    print('NOTE: outside Colab; checkpoint directory is LOCAL, not Google Drive.',ROOT)
RUN_DIR = os.path.join(ROOT, 'Simulation2', 'full' if RUN_FULL else 'diagnostic')
os.makedirs(RUN_DIR,exist_ok=True)
CPATH = os.path.join(RUN_DIR, 'checkpoint.pkl')
config = dict(version='FixedRank_HierarchicalBeta_RiskSensitive_PT_SIM2_v6_targeted_beta_tail_mh_exact', simulations=SIMULATIONS, full=RUN_FULL,
              risk_xi=RISK_XI, n_chains=N_CHAINS, iters=(MCMC_BASE,MCMC_EXTEND,MCMC_MAX),
              burn=MCMC_BURN, thin=MCMC_THIN, rhat_limit=RHAT_LIMIT, ess=(ESS_BULK_MIN,ESS_TAIL_MIN),
              hyperprior=dict(a0=2.,b0=2.,pool=4.,ak=2.,bk=1.), s1_prior=(2.,1.),
              gamma_sd=2.5,mu_sd=2.5, sigma_ig=(2.5,0.02), zeta=0.0015,
              training=dict(ref_lr=.005,red_lr=.005,red_weight_decay=.0005),
              checkpoint_seconds=CHECKPOINT_SECONDS, stop_on_first_failure=STOP_ON_FIRST_FAILURE,
              pt=(PT_TEMPS,PT_MIN_BETA,PT_SWAP_EVERY,PT_ADAPT_EVERY,PT_GLOBAL_MOVE_EVERY,PT_PRIOR_REFRESH_EVERY),
              hot_sweep_every=PT_HOT_SWEEP_EVERY,hot_predictors=PT_HOT_PREDICTORS,
              spectral_block_every=PT_SPECTRAL_BLOCK_EVERY,coupled_tail_every=PT_COUPLED_TAIL_EVERY,
               tail_mh=(TAIL_PRIOR_MH_EVERY,TAIL_PRIOR_MH_CUTOFFS,TAIL_PRIOR_MH_WEIGHTS,
                        TAIL_PRIOR_MH_INDEP_PROB,TAIL_PRIOR_MH_RW_SCALES,TAIL_RATIO_SLICE_WIDTH,
                        tuple((j,tuple(inds)) for j,inds in TAIL_FOCUS.items()),TAIL_PRIORITY))
SIGNATURE = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
# A different invariant MCMC transition schedule deserves a fresh, clearly labelled
# diagnostic run. Data, teacher and fitted reference parameters MAY be reused.
# Do not pool posterior traces from the previous failed V5 sampler with V6.
_BASE_ROOT = ('/content/drive/MyDrive/JRSSB_Embedding_Allocation' if os.path.isdir('/content/drive/MyDrive')
              else os.path.abspath('./JRSSB_Embedding_Allocation'))
SOURCE_ROOTS = (
    ('V5',os.path.join(_BASE_ROOT,'FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V5_DENSE_TAIL')),
    ('V4',os.path.join(_BASE_ROOT,'FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V4_FUSED')),
)
REQUIRED_FIT = ('task','name','seed','truth','teacher','training','validation','test','ref_params',
                'reference_train_mse','reference_test_mse')
def recoverable_fit_sources():
    paths=[]
    for version,root in SOURCE_ROOTS:
        directory=os.path.join(root,'Simulation2','full' if RUN_FULL else 'diagnostic')
        trial=[os.path.join(directory,'checkpoint.pkl')]
        if os.path.isdir(directory):
            for entry in sorted(os.listdir(directory),reverse=True):
                arc=os.path.join(directory,entry,'checkpoint.pkl')
                if entry.startswith('archive_') and os.path.isfile(arc): trial.append(arc)
        for path in trial:
            if os.path.isfile(path):
                try:
                    with open(path,'rb') as f: candidate=pickle.load(f)
                    work=candidate.get('work',{})
                    if all(q in work for q in REQUIRED_FIT) and str(work['task'][0])=='2':
                        paths.append((version,path,work))
                except Exception as exc:
                    print('Unable to examine legacy checkpoint',path,repr(exc),flush=True)
    return paths
_FIT_SOURCES=recoverable_fit_sources()
print('\nRUN SELECTION |',RUN_DIR)
print('[1] START FRESH (new simulated data and reference network)')
print('[2] REUSE V5/V4 DATA+REFERENCE; RESTART all four V6 MCMC ladders')
print('[3] RESUME V6 CHECKPOINT (same configuration)')
if _FIT_SOURCES:
    print('Available fitted-reference imports:',[(v,p) for v,p,_ in _FIT_SOURCES],flush=True)
else:
    print('No recoverable V5/V4 fit in checkpoint files; use option 1. (Old failed run may have cleared its work state.)')
while True:
    MODE = input('Enter 1, 2 or 3: ').strip()
    if MODE=='1' or (MODE=='2' and _FIT_SOURCES) or (MODE=='3' and os.path.isfile(CPATH)):
        break
    print('Option 2 needs a recoverable legacy fitted-reference checkpoint; option 3 needs an existing V6 checkpoint.')
if MODE in ('1','2') and os.path.exists(CPATH):
    archived=os.path.join(RUN_DIR,'archive_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    os.makedirs(archived,exist_ok=True)
    shutil.move(CPATH,os.path.join(archived,'checkpoint.pkl'))
    print('Previous V6 checkpoint archived:',archived)
if MODE=='3':
    with open(CPATH,'rb') as f: STATE=pickle.load(f)
    if STATE.get('signature')!=SIGNATURE:
        raise ValueError('V6 checkpoint signature mismatch: use the original configuration or START FRESH.')
    print('V6 RESUMED:',STATE.get('task_name'),'| phase:',STATE.get('work',{}).get('phase'))
elif MODE=='2':
    v,path,oldw=_FIT_SOURCES[0]
    freshw={key:oldw[key] for key in REQUIRED_FIT}
    freshw['phase']='mcmc'
    STATE=dict(signature=SIGNATURE,completed={},work=freshw,created=datetime.now().isoformat(),
               task_name=freshw['name'])
    print(f'REUSED {v} DATA/REFERENCE FROM {path}. MCMC draws/replicas reset to zero.',flush=True)
else:
    STATE=dict(signature=SIGNATURE,completed={},work={},created=datetime.now().isoformat(),task_name=None)
LAST_SAVE = time.monotonic()

def atomic_checkpoint(force=False):
    global LAST_SAVE
    if not force and (time.monotonic()-LAST_SAVE)<CHECKPOINT_SECONDS:return
    tmp=CPATH+'.tmp'
    with open(tmp,'wb') as f:
        pickle.dump(STATE,f,protocol=pickle.HIGHEST_PROTOCOL)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp,CPATH)
    LAST_SAVE=time.monotonic()
    print('  [DRIVE CHECKPOINT]',datetime.now().strftime('%H:%M:%S'), '| phase:',STATE.get('work',{}).get('phase'), flush=True)

if MODE=='2':
    atomic_checkpoint(force=True)  # new V6 state; V5/V4 untouched

def seed_all(seed):
    np.random.seed(seed);random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def orthogonal(k,rng):
    q,r=np.linalg.qr(rng.normal(size=(k,k)))
    return q * np.where(np.diag(r)>=0,1.,-1.)

def centered_orth(n,k,rng):
    a=rng.normal(size=(n,k));a-=a.mean(axis=0,keepdims=True)
    return np.linalg.qr(a)[0][:,:k]

def make_theta(n,pz,singular,rng):
    k=min(n-1,pz)
    t=centered_orth(n,k,rng) @ np.diag(np.asarray(singular[:k])) @ orthogonal(pz,rng)[:,:k].T
    return t-t.mean(axis=0,keepdims=True)

def setup(sim):
    if sim=='1':
        nlev=[8,10,12,9];pz=4;q=4;pu=2
        scen=[(f'{regime}_n{n}',regime,n) for regime in ('clear','weak') for n in (500,1000,2000)]
        if not RUN_FULL:scen=[x for x in scen if x[2]==1000]
        budgets=[sum(nlev),60,80,100,120,140,pz*sum(nlev)];num_reps=100 if RUN_FULL else 1
    else:
        nlev=[3,5,8,4,10,6,7,12,5,9,4,11];pz=5;q=5;pu=3
        scen=[(r,r,1200) for r in ('heterogeneous','near_homogeneous')]
        budgets=[sum(nlev),125,170,215,255,300,340,sum(x*min(x-1,pz) for x in nlev)]
        num_reps=50 if RUN_FULL else 1
    klist=[min(n-1,pz) for n in nlev]
    bases=[]
    for n in nlev:
        _,_,vt=np.linalg.svd(np.ones((1,n)),full_matrices=True)
        bases.append(vt[1:,:].T.copy())
    return dict(sim=sim,nlev=nlev,m=len(nlev),pz=pz,q=q,pu=pu,klist=klist,bases=bases,
                scenarios=scen,budgets=sorted(set(budgets)),num_reps=num_reps,
                n_val=500,n_test=(3000 if sim=='1' else 4000), noise_sd=.1,
                ref_epochs=400 if RUN_FULL else 250,red_epochs=300 if RUN_FULL else 220,
                ref_restarts=5 if RUN_FULL else 3,red_restarts=5 if RUN_FULL else 3)

CFG=None

def teacher(sim):
    rng=np.random.default_rng(8877 if sim=='1' else 90210)
    p=CFG['pz']
    if sim=='1':b1=np.array([.1,-.1,.05,-.05]);b2=np.array([.05,-.05,.05,-.05])
    else:b1=np.array([.1,-.05,.05,-.1,.05]);b2=np.array([.05,-.05,.05,0,-.05])
    return [(orthogonal(p,rng),b1),(orthogonal(p,rng),b2)]

def target(regime):
    c=CFG; sim=c['sim'];m=c['m'];nlev=c['nlev'];klist=c['klist']
    rng=np.random.default_rng(12345 if sim=='1' else 4567)
    if sim=='1':
        raw=([[1.8,.3,.1,.05],[1.6,.7,.2,.05],[2.,1.,.4,.1],[1.5,.5,.2,.05]] if regime=='clear'
             else [[1.8,1.2,.9,.6],[1.6,1.3,1.,.7],[2.,1.6,1.3,1.],[1.5,1.1,.85,.6]])
        spec=[np.array(raw[j][:klist[j]]) for j in range(m)]
        g=rng.normal(scale=.2,size=(c['pu'],c['pz']));mu=np.array([.1,-.1,.05,-.05]);dtrue=None
    else:
        dims={'heterogeneous':(1,4,2,1,4,2,3,1,2,4,1,3),
              'near_homogeneous':(2,)*12}[regime]
        strong=np.array([1.35,1.1,.85,.6,.45]);spec=[]
        for j in range(m):
            sc=np.sqrt(nlev[j]/np.median(nlev))*(1.+.015*(j-(m-1)/2))
            sj=np.zeros(klist[j]);sj[:dims[j]]=strong[:dims[j]]*sc;spec.append(sj)
        g=rng.normal(scale=.15,size=(c['pu'],c['pz']));mu=np.array([.1,-.05,.05,.05,-.05]);dtrue=dims
    t=[make_theta(nlev[j],c['pz'],spec[j],rng) for j in range(m)]
    return dict(theta=t,gamma=g,mu=mu,spec=spec,dtrue=dtrue)

def phi(z,weights):
    (w1,b1),(w2,b2)=weights
    return np.tanh(z @ w1.T + b1) @ w2.T + b2

def data(n,truth,weights,seed):
    c=CFG;rng=np.random.default_rng(seed)
    C=[rng.integers(0,c['nlev'][j],size=n) for j in range(c['m'])]
    U=rng.normal(size=(n,c['pu']))
    z=np.broadcast_to(truth['mu'],(n,c['pz'])).copy()+U@truth['gamma']
    for j in range(c['m']):z+=truth['theta'][j][C[j]]
    mean=phi(z,weights);Y=mean+c['noise_sd']*rng.normal(size=mean.shape)
    return C,U,Y

class Reference(nn.Module):
    def __init__(self):
        super().__init__();c=CFG
        self.emb=nn.ModuleList([nn.Embedding(n,c['pz']) for n in c['nlev']])
        for e in self.emb:nn.init.normal_(e.weight,std=.1 if c['sim']=='1' else .05)
        self.gamma=nn.Parameter(.08*torch.randn(c['pu'],c['pz']));self.mu=nn.Parameter(torch.zeros(c['pz']))
        self.l1=nn.Linear(c['pz'],c['pz']);self.l2=nn.Linear(c['pz'],c['q'])
        with torch.no_grad():
            for l in (self.l1,self.l2): l.weight.copy_(torch.linalg.qr(torch.randn(*l.weight.shape))[0]);l.bias.zero_()
    def forward(self,C,U):
        z=self.mu+U@self.gamma
        for j,e in enumerate(self.emb):z=z+e(C[j])
        return self.l2(torch.tanh(self.l1(z)))
    @torch.no_grad()
    def project(self):
        for e in self.emb:e.weight-=e.weight.mean(0,keepdim=True)
        for l in (self.l1,self.l2):
            a,_,v=torch.linalg.svd(l.weight,full_matrices=False);l.weight.copy_(a@v)

class Reduced(nn.Module):
    def __init__(self,d):
        super().__init__();c=CFG;self.d=d
        self.emb=nn.ModuleList([nn.Embedding(c['nlev'][j],d[j]) for j in range(c['m'])])
        for e in self.emb:nn.init.normal_(e.weight,std=.1)
        self.gamma=nn.Parameter(.08*torch.randn(c['pu'],c['pz']));self.mu=nn.Parameter(torch.zeros(c['pz']))
        self.l1=nn.Linear(sum(d)+c['pz'],c['pz']);self.l2=nn.Linear(c['pz'],c['q'])
    def forward(self,C,U):
        z=torch.cat([e(C[j]) for j,e in enumerate(self.emb)]+[self.mu+U@self.gamma],dim=1)
        return self.l2(torch.tanh(self.l1(z)))
    @torch.no_grad()
    def project(self):
        for e in self.emb:e.weight-=e.weight.mean(0,keepdim=True)

def tensors(t):
    C,U,Y=t
    return ([torch.as_tensor(z,dtype=torch.long,device=DEVICE) for z in C],
            torch.as_tensor(U,dtype=torch.float32,device=DEVICE),
            torch.as_tensor(Y,dtype=torch.float32,device=DEVICE))

def fit_model(train,kind,seed,d=None):
    c=CFG;C,U,Y=tensors(train)
    best=None;bestloss=np.inf
    restarts=c['ref_restarts'] if kind=='reference' else c['red_restarts']
    epochs=c['ref_epochs'] if kind=='reference' else c['red_epochs']
    for r in range(restarts):
        seed_all(seed+1000*r)
        net=(Reference() if kind=='reference' else Reduced(d)).to(DEVICE)
        opt=torch.optim.Adam(net.parameters(),lr=.005,weight_decay=5e-5 if kind=='reference' else 5e-4)
        for ep in range(epochs):
            opt.zero_grad();loss=F.mse_loss(net(C,U),Y);loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),5.);opt.step();net.project()
            if ep%50==0:atomic_checkpoint()
        with torch.no_grad():result=float(F.mse_loss(net(C,U),Y).item())
        if result<bestloss:
            bestloss=result;best=net
        else:del net
    print(f'  {kind} d={d} training MSE={bestloss:.6f}',flush=True)
    return best,bestloss

def predict(net,dt):
    C,U,Y=tensors(dt)
    with torch.no_grad():return net(C,U).detach().cpu().numpy()

def reference_parameters(net):
    with torch.no_grad():
        th=[e.weight.detach().cpu().numpy().astype(float).copy() for e in net.emb]
        th=[v-v.mean(0,keepdims=True) for v in th]
        g=net.gamma.detach().cpu().numpy().astype(float).copy()
        mu=net.mu.detach().cpu().numpy().astype(float).copy()
        w=[(l.weight.detach().cpu().numpy().astype(float).copy(),l.bias.detach().cpu().numpy().astype(float).copy())
           for l in (net.l1,net.l2)]
    return dict(theta=th,gamma=g,mu=mu,omega=w)

def mse(y,yhat):return float(np.mean((y-yhat)**2))

# ---- Fixed-dimension posterior with integrated inverse-Gamma noise variance ----
S1A,S1B=2.,1.
A0,B0,POOL,AK,BK=2.,2.,4.,2.,1.
TAUG,TAUM=2.5,2.5
SIGMA_A,SIGMA_B=2.5,.02

def cliplogit(p):
    p=np.clip(p,1e-12,1-1e-12)
    return np.log(p)-np.log1p(-p)

def log_beta_logit(v,a,b):
    # Correct Jacobian: log Beta density on rho plus log |d rho / d v|.
    if a<=0 or b<=0:return -np.inf
    return float(a*log_expit(v)+b*log_expit(-v)-betaln(a,b))

def log_gamma_log(v,a,b):
    if abs(v)>50:return -np.inf
    return float(a*v-b*np.exp(v)+a*np.log(b)-gammaln(a))

def qr_haar(a):
    q,r=np.linalg.qr(a,mode='reduced')
    return q*np.where(np.diag(r)>=0.,1.,-1.)

def theta_from(s,j,refresh=None):
    # Computational cache ONLY: deterministic QR orientations depend solely on A,B.
    # The cache has no prior, likelihood, or statistical influence; always refresh
    # immediately when a proposed update changes the matching auxiliary matrix.
    c=CFG;K=c['klist'][j]
    if refresh in ('A','AB') or '_U' not in s:
        s['_U']=qr_haar(s['A'])[:,:K]
    if refresh in ('B','AB') or '_V' not in s:
        s['_V']=qr_haar(s['B'])[:,:K]
    u=s['_U'];v=s['_V']
    eig=np.empty(K);eig[0]=np.exp(s['x'])
    for h in range(1,K):eig[h]=eig[h-1]*expit(s['eta'][h-1])
    t=(c['bases'][j]@u)@np.diag(np.sqrt(eig))@v.T
    return t-t.mean(0,keepdims=True),eig

def start_spectrum(theta,j,rng,spread=1.0):
    c=CFG;K=c['klist'][j]
    u,sv,vt=np.linalg.svd(theta-theta.mean(0),full_matrices=False)
    s2=np.maximum(sv[:K]**2,1e-7)
    s2=np.minimum.accumulate(s2)
    A=c['bases'][j].T@u[:,:K];B=vt.T[:,:K]
    A=A+(0.05*spread)*rng.normal(size=A.shape);B=B+(0.05*spread)*rng.normal(size=B.shape)
    return dict(A=A,B=B,x=float(np.log(s2[0])+rng.normal(scale=.15*spread)),
                eta=cliplogit(np.clip(s2[1:]/s2[:-1],.001,.999))+rng.normal(scale=.3*spread,size=K-1))
def loglik(Y, yh):
    resid=Y-yh
    return float(-(SIGMA_A+.5*Y.size)*np.log(SIGMA_B+.5*np.sum(resid*resid)))

# JIT compiles ONLY deterministic likelihood arithmetic. Statistical updates,
# priors, exact slice kernels, replica-exchange probabilities and RNG stay Python.
# Neither fused routine changes the unnormalized integrated-variance target.
try:
    from numba import njit
    _NUMBA_AVAILABLE=True
except Exception:
    _NUMBA_AVAILABLE=False
    njit=None
_FAST_LL_MODE='numpy'
_FAST_LL_CHECKED=False
if _NUMBA_AVAILABLE:
    @njit(cache=False,fastmath=False)
    def _fused_ll_z(z,Y,w1,b1,w2,b2,alpha,beta):
        n,p=z.shape; hidden=np.empty(w1.shape[0]);ss=0.0
        for i in range(n):
            for h in range(w1.shape[0]):
                t=b1[h]
                for r in range(p): t+=z[i,r]*w1[h,r]
                hidden[h]=math.tanh(t)
            for q in range(w2.shape[0]):
                pred=b2[q]
                for h in range(w1.shape[0]): pred+=hidden[h]*w2[q,h]
                dif=Y[i,q]-pred;ss+=dif*dif
        return -(alpha+0.5*Y.size)*math.log(beta+0.5*ss)

    @njit(cache=False,fastmath=False)
    def _fused_ll_base(base,theta,cidx,Y,w1,b1,w2,b2,alpha,beta):
        n,p=base.shape; hidden=np.empty(w1.shape[0]);ss=0.0
        for i in range(n):
            row=cidx[i]
            for h in range(w1.shape[0]):
                t=b1[h]
                for r in range(p): t+=(base[i,r]+theta[row,r])*w1[h,r]
                hidden[h]=math.tanh(t)
            for q in range(w2.shape[0]):
                pred=b2[q]
                for h in range(w1.shape[0]): pred+=hidden[h]*w2[q,h]
                dif=Y[i,q]-pred;ss+=dif*dif
        return -(alpha+0.5*Y.size)*math.log(beta+0.5*ss)

def ll_from_z(z,Y,omega):
    if _FAST_LL_MODE=='numba':
        return float(_fused_ll_z(z,Y,*_as_weights(omega),SIGMA_A,SIGMA_B))
    return loglik(Y,phi(z,omega))

def ll_from_base(base,th,Cj,Y,omega,base_hidden=None):
    if _FAST_LL_MODE=='numba':
        return float(_fused_ll_base(base,th,Cj,Y,*_as_weights(omega),SIGMA_A,SIGMA_B))
    if base_hidden is not None:
        (w1,b1),(w2,b2)=omega
        predicted=np.tanh(base_hidden+(th@w1.T)[Cj])@w2.T+b2
        return loglik(Y,predicted)
    return loglik(Y,phi(base+th[Cj],omega))

def _as_weights(omega):
    (w1,b1),(w2,b2)=omega
    return w1,b1,w2,b2

def configure_fast_likelihood(train,omega):
    # Validate BOTH fused forms against the unmodified NumPy likelihood,
    # at the actual training size, then use JIT only when locally faster.
    global _FAST_LL_MODE,_FAST_LL_CHECKED
    if _FAST_LL_CHECKED: return
    _FAST_LL_CHECKED=True
    if not _NUMBA_AVAILABLE:
        print('FAST LL: Numba not installed; using preprojected cached NumPy likelihood.',flush=True)
        return
    import time as _clock
    C,U,Y=train
    rng=np.random.default_rng(47017)
    z=rng.normal(scale=.5,size=(Y.shape[0],CFG['pz']))
    th=rng.normal(scale=.05,size=(CFG['nlev'][0],CFG['pz']))
    # Warm JIT before timing; no random seed/state of the sampler is touched.
    w=_as_weights(omega)
    a=_fused_ll_z(z,Y,*w,SIGMA_A,SIGMA_B)
    b=_fused_ll_base(z,th,C[0],Y,*w,SIGMA_A,SIGMA_B)
    a0=loglik(Y,phi(z,omega))
    b0=loglik(Y,phi(z+th[C[0]],omega))
    if not np.isclose(a,a0,rtol=0,atol=5e-7) or not np.isclose(b,b0,rtol=0,atol=5e-7):
        print('FAST LL: validation failed, retaining NumPy evaluator.',a-a0,b-b0,flush=True)
        return
    N=20
    t=_clock.perf_counter()
    for _ in range(N):
        loglik(Y,phi(z+th[C[0]],omega))
    slow=(_clock.perf_counter()-t)/N
    bh=z@omega[0][0].T+omega[0][1]
    t=_clock.perf_counter()
    for _ in range(N):
        ll_from_base(z,th,C[0],Y,omega,bh)
    numpy_cached=(_clock.perf_counter()-t)/N
    t=_clock.perf_counter()
    for _ in range(N):
        _fused_ll_base(z,th,C[0],Y,*w,SIGMA_A,SIGMA_B)
    fast=(_clock.perf_counter()-t)/N
    # Prefer the fastest VERIFIED evaluator at this machine's actual n=1200.
    if fast<0.98*numpy_cached:
        _FAST_LL_MODE='numba'
    print(f'FAST LL: backend={_FAST_LL_MODE} | original {slow*1e6:.1f}us, '
          f'cached NumPy {numpy_cached*1e6:.1f}us, compiled {fast*1e6:.1f}us per candidate.',flush=True)

def draw_sigma2(Y,yh,rng):
    a=SIGMA_A+.5*Y.size;b=SIGMA_B+.5*np.sum((Y-yh)**2)
    return float(1./rng.gamma(a,1./b))

def ess_block(current,log_L,rng,prior_sd=1.,max_evals=300):
    # Elliptical slice; valid for zero-mean isotropic Gaussian prior coordinates.
    nu=prior_sd*rng.normal(size=current.shape)
    level=log_L(current)+np.log(rng.random())
    angle=rng.uniform(0.,2*np.pi);low=angle-2*np.pi;high=angle
    for evaluations in range(1,max_evals+1):
        proposed=current*np.cos(angle)+nu*np.sin(angle)
        if log_L(proposed)>=level:return proposed,evaluations
        if angle<0:low=angle
        else:high=angle
        angle=rng.uniform(low,high)
    raise RuntimeError('Elliptical slice exceeded bracket budget; stop and inspect sampler.')

def slice1(x,ld,rng,w=.5,max_evals=400):
    current=ld(x)
    if not np.isfinite(current):raise FloatingPointError('Nonfinite current log density in slice sampler.')
    level=current+np.log(rng.random());offset=rng.random()
    L=x-w*offset;R=L+w
    J=int(rng.integers(20));K=19-J; ne=0
    while J>0 and ld(L)>level:L-=w;J-=1;ne+=1
    while K>0 and ld(R)>level:R+=w;K-=1;ne+=1
    for t in range(max_evals):
        xnew=rng.uniform(L,R);v=ld(xnew);ne+=1
        if v>=level:return float(xnew),ne
        if xnew<x:L=xnew
        else:R=xnew
    raise RuntimeError('Scalar slice exceeded shrinkage budget.')

def current_hyper(h):
    return expit(h['z0']),expit(h['zj']),np.exp(h['lk'])

def hierarchical_logprior(h,states):
    means0,means,kappa=current_hyper(h)
    if not np.isfinite(kappa) or kappa>1e5:return -np.inf
    lp=log_beta_logit(h['z0'],A0,B0)+log_gamma_log(h['lk'],AK,BK)
    for j in range(CFG['m']):
        lp+=log_beta_logit(h['zj'][j],POOL*means0,POOL*(1-means0))
        a=kappa*means[j];b=kappa*(1-means[j])
        for eta in states[j]['eta']:
            lp+=log_beta_logit(eta,a,b)
    return float(lp)

# Faster exact conditional log densities: terms constant in the coordinate
# under update are omitted, as permitted by slice sampling. No approximation.
def local_zj_logprior(v,j,h,states):
    h0=expit(h['z0']);rho=expit(v);kappa=np.exp(h['lk'])
    if not np.isfinite(kappa) or kappa>1e5:return -np.inf
    a=kappa*rho;b=kappa*(1.-rho)
    et=states[j]['eta']
    return float(log_beta_logit(v,POOL*h0,POOL*(1.-h0))+
                 np.sum(a*log_expit(et)+b*log_expit(-et))-len(et)*betaln(a,b))

def local_z0_logprior(v,h):
    rho0=expit(v)
    zj=h['zj']
    aa=POOL*rho0;bb=POOL*(1.-rho0)
    return float(log_beta_logit(v,A0,B0)+
                 np.sum(aa*log_expit(zj)+bb*log_expit(-zj))-len(zj)*betaln(aa,bb))

def local_kappa_logprior(v,h,states):
    if abs(v)>50:return -np.inf
    kappa=np.exp(v)
    if not np.isfinite(kappa) or kappa>1e5:return -np.inf
    means=expit(h['zj'])
    lp=log_gamma_log(v,AK,BK)
    for j in range(CFG['m']):
        et=states[j]['eta'];a=kappa*means[j];b=kappa*(1.-means[j])
        lp+=float(np.sum(a*log_expit(et)+b*log_expit(-et))-len(et)*betaln(a,b))
    return float(lp)

def start_chain(ref,train,seed,spread=1.0):
    C,U,Y=train;rng=np.random.default_rng(seed)
    s=[start_spectrum(ref['theta'][j],j,rng,spread=spread) for j in range(CFG['m'])]
    th=[theta_from(s[j],j)[0] for j in range(CFG['m'])]
    g=ref['gamma'].copy()+(0.025*spread)*rng.normal(size=ref['gamma'].shape)
    mu=ref['mu'].copy()+(0.025*spread)*rng.normal(size=ref['mu'].shape)
    z=mu+U@g
    for j in range(CFG['m']):z+=th[j][C[j]]
    yh=phi(z,ref['omega']);ll=loglik(Y,yh)
    h=dict(z0=float(cliplogit(rng.beta(2,2))),
           zj=cliplogit(np.clip(rng.beta(4,4,size=CFG['m']),.001,.999)),
           lk=float(np.log(rng.gamma(2,1))))
    return dict(it=0,rng=rng,s=s,th=th,g=g,mu=mu,z=z,yh=yh,ll=ll,h=h,
                traces={'s2':[],'bar':[],'bar0':[],'kappa':[],'sigma2':[],'ll':[],
                        'gamma':[],'mu':[]},
                effort=np.zeros((CFG['m'],3),dtype=np.int64),
                updates=np.zeros((CFG['m'],3),dtype=np.int64),
                h_effort=np.zeros(3,dtype=np.int64),h_updates=np.zeros(3,dtype=np.int64),
                joint_effort=0,joint_updates=0,global_effort=0,global_updates=0,
                refresh_try=np.zeros(CFG['m'],dtype=np.int64),refresh_accept=np.zeros(CFG['m'],dtype=np.int64),
                 tail_mh_try=np.zeros((CFG['m'],max(CFG['klist'])-1),dtype=np.int64),
                 tail_mh_accept=np.zeros((CFG['m'],max(CFG['klist'])-1),dtype=np.int64),
                 tail_mh_type_try=np.zeros(2,dtype=np.int64),
                 tail_mh_type_accept=np.zeros(2,dtype=np.int64),
                spectral_block_evals=np.zeros(CFG['m'],dtype=np.int64),
                spectral_block_updates=np.zeros(CFG['m'],dtype=np.int64),
                coupled_tail_evals=np.zeros(CFG['m'],dtype=np.int64),
                coupled_tail_updates=np.zeros(CFG['m'],dtype=np.int64))
def sampler_step(p,train,omega,beta=1.0,record=True,global_move=False,
                  prior_refresh=False,spectral_block=False,coupled_tail=False,predictor_subset=None,
                 targeted_tail_mh=False):
    # Every component below is an exact conditional/ESS/MH update for the
    # same beta-tempered posterior. A hot predictor_subset merely composes
    # fewer invariant kernels; no pseudo-likelihood or approximate gradients.
    C,U,Y=train;rng=p['rng']
    active=range(CFG['m']) if predictor_subset is None else predictor_subset
    for j in active:
        sj=p['s'][j];Cj=C[j]
        base_z=p['z']-p['th'][j][Cj]
        (w1,b1),_=omega
        base_hidden=base_z@w1.T+b1 if _FAST_LL_MODE=='numpy' else None
        # While updating this predictor the contribution from every OTHER
        # predictor is fixed; update latent state only after finishing this j.
        for bi,key in enumerate(('A','B')):
            old=sj[key].copy();accepted={}
            def evaluate(a):
                sj[key]=a
                th,_=theta_from(sj,j,refresh=key)
                ll=ll_from_base(base_z,th,Cj,Y,omega,base_hidden)
                accepted.update(th=th,ll=ll)
                return beta*ll
            new,ne=ess_block(old,evaluate,rng)
            sj[key]=new
            p['ll']=accepted['ll'];p['th'][j]=accepted['th']
            p['effort'][j,bi]+=ne;p['updates'][j,bi]+=1
        _,means,kappa=current_hyper(p['h'])
        for idx in range(CFG['klist'][j]):
            is_x=(idx==0);ii=idx-1
            old=float(sj['x'] if is_x else sj['eta'][ii])
            accepted={}
            def score(value):
                if not np.isfinite(value) or (is_x and abs(value)>60) or ((not is_x) and abs(value)>700):return -np.inf
                if is_x:sj['x']=value
                else:sj['eta'][ii]=value
                th,_=theta_from(sj,j)
                ll=ll_from_base(base_z,th,Cj,Y,omega,base_hidden)
                accepted.update(th=th,ll=ll)
                prior=log_gamma_log(value,S1A,S1B) if is_x else log_beta_logit(value,kappa*means[j],kappa*(1-means[j]))
                return prior+beta*ll
            new,ne=slice1(old,score,rng,w=.25 if is_x else (TAIL_RATIO_SLICE_WIDTH if (j in TAIL_FOCUS and ii in TAIL_FOCUS[j]) else .7))
            if is_x:sj['x']=new
            else:sj['eta'][ii]=new
            p['th'][j]=accepted['th'];p['ll']=accepted['ll']
            p['effort'][j,2]+=ne;p['updates'][j,2]+=1
        # Reconstruct z ONCE per predictor, rather than after every ESS/slice.
        p['z']=base_z+p['th'][j][Cj]
    if prior_refresh:
        _,means,kappa=current_hyper(p['h'])
        for j in active:
            sj=p['s'][j];old_eta=sj['eta'].copy()
            if old_eta.size==0:continue
            aa=max(1e-7,kappa*means[j]);bb=max(1e-7,kappa*(1-means[j]))
            rho_prop=np.clip(rng.beta(aa,bb,size=old_eta.size),1e-12,1-1e-12)
            sj['eta']=cliplogit(rho_prop)
            th,_=theta_from(sj,j)
            base_z=p['z']-p['th'][j][C[j]]
            bh=base_z@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
            new_ll=ll_from_base(base_z,th,C[j],Y,omega,bh)
            p['refresh_try'][j]+=1
            if np.log(rng.random())<min(0.,beta*(new_ll-p['ll'])):
                p['refresh_accept'][j]+=1
                p['z']=base_z+th[C[j]];p['th'][j]=th;p['ll']=new_ll
            else:sj['eta']=old_eta
    # EXACT JOINT BETA-RATIO ELLIPTICAL SLICE:
    # Conditional Beta(a,b) ratios are represented as eta=logit(BetaPPF(Phi(t))),
    # with independent standard Gaussian t coordinates. ESS therefore preserves
    # the same hierarchical prior (and beta*collapsed likelihood).
    if spectral_block:
        _,means,kappa=current_hyper(p['h'])
        for j in active:
            sj=p['s'][j];neta=len(sj['eta'])
            if not neta: continue
            aa=max(1e-8,kappa*means[j]);bb=max(1e-8,kappa*(1.-means[j]))
            rho0=np.clip(expit(sj['eta']),1e-13,1.-1e-13)
            initial=ndtri(np.clip(betainc(aa,bb,rho0),1e-13,1.-1e-13))
            base=p['z']-p['th'][j][C[j]]
            hidden=base@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
            cache={}
            def transform(t):
                u=np.clip(ndtr(t),1e-13,1.-1e-13)
                return cliplogit(np.clip(betaincinv(aa,bb,u),1e-13,1.-1e-13))
            def f(t):
                sj['eta']=transform(t)
                th,_=theta_from(sj,j)
                ll=ll_from_base(base,th,C[j],Y,omega,hidden)
                cache.update(theta=th,ll=ll)
                return beta*ll
            nextg,ne=ess_block(initial,f,rng,max_evals=500)
            sj['eta']=transform(nextg)
            p['th'][j]=cache['theta'];p['ll']=cache['ll']
            p['z']=base+p['th'][j][C[j]]
            p['spectral_block_evals'][j]+=ne;p['spectral_block_updates'][j]+=1

    # EXACT COUPLED WEAK-SPECTRUM MOVE:
    # Combine Gaussian A[:,k], B[:,k], and Gaussianized Beta ratio eta[k-1]
    # in one prior-preserving elliptical slice update. Particularly targets
    # trailing spectral components with strongly coupled directions and ratios.
    if coupled_tail:
        _,means,kappa=current_hyper(p['h'])
        for j in active:
            sj=p['s'][j];K=CFG['klist'][j]
            if K<2:continue
            k=int(K-1 if rng.random()<.65 else rng.integers(1,K))
            aa=max(1e-8,kappa*means[j]);bb=max(1e-8,kappa*(1.-means[j]))
            old_u=float(np.clip(betainc(aa,bb,np.clip(expit(sj['eta'][k-1]),1e-13,1.-1e-13)),1e-13,1.-1e-13))
            na=sj['A'].shape[0];nb=sj['B'].shape[0]
            initial=np.r_[sj['A'][:,k],sj['B'][:,k],float(ndtri(old_u))]
            base=p['z']-p['th'][j][C[j]]
            hidden=base@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
            cache={}
            def f(t):
                sj['A'][:,k]=t[:na];sj['B'][:,k]=t[na:na+nb]
                u=float(np.clip(ndtr(t[-1]),1e-13,1.-1e-13))
                rho=float(np.clip(betaincinv(aa,bb,u),1e-13,1.-1e-13))
                sj['eta'][k-1]=float(cliplogit(rho))
                th,_=theta_from(sj,j,refresh='AB')
                ll=ll_from_base(base,th,C[j],Y,omega,hidden)
                cache.update(theta=th,ll=ll)
                return beta*ll
            nxt,ne=ess_block(initial,f,rng,max_evals=500)
            # Explicitly synchronize Gaussian auxiliaries and cached QR factors
            # with the actually accepted point before modifying the latent state.
            sj['A'][:,k]=nxt[:na];sj['B'][:,k]=nxt[na:na+nb]
            uj=float(np.clip(ndtr(nxt[-1]),1e-13,1.-1e-13))
            sj['eta'][k-1]=float(cliplogit(np.clip(betaincinv(aa,bb,uj),1e-13,1.-1e-13)))
            p['th'][j]=cache['theta'];p['ll']=cache['ll']
            p['z']=base+p['th'][j][C[j]]
            p['coupled_tail_evals'][j]+=ne;p['coupled_tail_updates'][j]+=1

    oldg=p['g'].copy();oldmu=p['mu'].copy()
    ng=oldg.size;joint_old=np.concatenate([oldg.reshape(-1)/TAUG,oldmu/TAUM])
    accepted={}
    def joint_ll(a):
        candg=(a[:ng]*TAUG).reshape(oldg.shape);candmu=a[ng:]*TAUM
        zz=p['z']+U@(candg-oldg)+(candmu-oldmu)
        ll=ll_from_z(zz,Y,omega)
        accepted.update(z=zz,ll=ll)
        return beta*ll
    joint_new,joint_ne=ess_block(joint_old,joint_ll,rng,prior_sd=1.)
    p['g']=(joint_new[:ng]*TAUG).reshape(oldg.shape)
    p['mu']=joint_new[ng:]*TAUM
    p['z']=accepted['z'];p['ll']=accepted['ll']
    p['joint_effort']+=joint_ne;p['joint_updates']+=1
    if global_move:
        j=int(rng.integers(CFG['m']));sj=p['s'][j]
        oldA=sj['A'].copy();oldB=sj['B'].copy()
        oldg=p['g'].copy();oldmu=p['mu'].copy()
        na=oldA.size;nb=oldB.size;ng=oldg.size
        block_old=np.concatenate([oldA.reshape(-1),oldB.reshape(-1),oldg.reshape(-1)/TAUG,oldmu/TAUM])
        accepted={}
        base_z=p['z']-p['th'][j][C[j]]
        def block_ll(a):
            aa=a[:na].reshape(oldA.shape);bb=a[na:na+nb].reshape(oldB.shape)
            newg=(a[na+nb:na+nb+ng]*TAUG).reshape(oldg.shape)
            newmu=a[na+nb+ng:]*TAUM
            sj['A']=aa;sj['B']=bb
            th,_=theta_from(sj,j,refresh='AB')
            zz=base_z+th[C[j]]+U@(newg-oldg)+(newmu-oldmu)
            ll=ll_from_z(zz,Y,omega)
            accepted.update(th=th,z=zz,ll=ll)
            return beta*ll
        new,ne=ess_block(block_old,block_ll,rng,prior_sd=1.,max_evals=500)
        sj['A']=new[:na].reshape(oldA.shape);sj['B']=new[na:na+nb].reshape(oldB.shape)
        p['g']=(new[na+nb:na+nb+ng]*TAUG).reshape(oldg.shape)
        p['mu']=new[na+nb+ng:]*TAUM
        p['z']=accepted['z'];p['th'][j]=accepted['th'];p['ll']=accepted['ll']
        p['global_effort']+=ne;p['global_updates']+=1
    h=p['h'];states=p['s']
    # Slice samplers only need a conditional log density up to a CONSTANT.
    # This avoids rescoring all 12 predictor priors for every hyper mean move.
    for typ in ('z0','zj','lk'):
        if typ=='zj':
            for j in range(CFG['m']):
                old=float(h['zj'][j])
                def ld(v):return local_zj_logprior(v,j,h,states)
                new,ne=slice1(old,ld,rng,w=.5)
                h['zj'][j]=new;p['h_effort'][1]+=ne;p['h_updates'][1]+=1
        else:
            old=float(h[typ])
            if typ=='z0':
                def ld(v):return local_z0_logprior(v,h)
            else:
                def ld(v):return local_kappa_logprior(v,h,states)
            new,ne=slice1(old,ld,rng,w=.3 if typ=='lk' else .5)
            h[typ]=new;i=0 if typ=='z0' else 2
            p['h_effort'][i]+=ne;p['h_updates'][i]+=1
    # Full-support, prior-corrected independent tail refresh BEFORE recording traces.
    if targeted_tail_mh:
        tail_mh_sweep(p,train,omega,beta,p['it']+1)
    # Cache exact predictions once per sweep, essential for sigma2 draw/checkpoint.
    p['yh']=phi(p['z'],omega)
    p['ll']=loglik(Y,p['yh'])
    p['it']+=1
    it=p['it']
    if record and it>MCMC_BURN and (it-MCMC_BURN)%MCMC_THIN==0:
        sp=np.array([np.pad(theta_from(p['s'][j],j)[1],(0,max(CFG['klist'])-CFG['klist'][j]),constant_values=np.nan)
                     for j in range(CFG['m'])])
        h0,means,kappa=current_hyper(p['h']);t=p['traces']
        t['s2'].append(sp)
        t['bar'].append(means.copy());t['bar0'].append(float(h0));t['kappa'].append(float(kappa))
        t['sigma2'].append(draw_sigma2(Y,p['yh'],rng));t['ll'].append(p['ll'])
        t['gamma'].append(p['g'].copy());t['mu'].append(p['mu'].copy())

# ---- Exact scalar Beta-ratio independence MH, targeted at near-zero tails ----
def tail_beta_mh_coordinate(p,train,omega,beta,j,k):
    """Exact mixture Metropolis-Hastings step for a hierarchical Beta tail ratio.

    One component proposes a new ratio from the complete conditional Beta
    PRIOR, three others from that same Beta law truncated at fixed scales.
    Mixture density corrections are included in the MH ratio. An alternative
    symmetric Gaussian random walk on logit(rho) is also MH corrected.
    Neither component alters the posterior, priors, or likelihood.
    """
    C,U,Y=train;sj=p['s'][j];rng=p['rng']
    if k<0 or k>=len(sj['eta']):return False
    _,means,kappa=current_hyper(p['h'])
    a=float(kappa*means[j]);b=float(kappa*(1.-means[j]))
    if a<=0 or b<=0 or not np.isfinite(a+b):raise ValueError('Invalid Beta shapes')
    old_eta=float(sj['eta'][k]);old_rho=float(expit(old_eta))
    if rng.random()<TAIL_PRIOR_MH_INDEP_PROB:
        cuts=TAIL_PRIOR_MH_CUTOFFS;weights=TAIL_PRIOR_MH_WEIGHTS
        comp=int(rng.choice(len(weights),p=weights))
        cs=[float(betainc(a,b,cut)) for cut in cuts]
        if any(not (0.<v<=1.) for v in cs):
            raise FloatingPointError('Beta CDF lost precision in multiscale MH proposal')
        if comp==0:
            new_rho=float(rng.beta(a,b))
        else:
            u=np.clip(rng.random()*cs[comp-1],np.nextafter(0.,1.),1.-1e-15)
            new_rho=float(betaincinv(a,b,u))
        new_rho=float(np.clip(new_rho,1e-300,1.-1e-15))
        new_eta=float(np.log(new_rho)-np.log1p(-new_rho))
        def factor(rho):
            return math.log(weights[0]+sum(weights[h+1]/cs[h] for h,cut in enumerate(cuts) if rho<=cut))
        # Beta prior and logit Jacobian cancel from target/proposal quotient.
        log_hastings=factor(old_rho)-factor(new_rho)
        move_type=0
    else:
        # The logit random-walk proposal is symmetric, so its q terms cancel.
        scale=TAIL_PRIOR_MH_RW_SCALES[int(rng.integers(len(TAIL_PRIOR_MH_RW_SCALES)))]
        new_eta=float(old_eta+rng.normal(scale=scale))
        if not np.isfinite(new_eta) or abs(new_eta)>700:return False
        new_rho=float(expit(new_eta))
        log_hastings=(log_beta_logit(new_eta,a,b)-log_beta_logit(old_eta,a,b))
        move_type=1
    base=p['z']-p['th'][j][C[j]]
    sj['eta'][k]=new_eta
    theta,_=theta_from(sj,j)
    hidden=base@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
    ll=ll_from_base(base,theta,C[j],Y,omega,hidden)
    p['tail_mh_try'][j,k]+=1
    p['tail_mh_type_try'][move_type]+=1
    if np.log(rng.random()) < min(0.,beta*(ll-p['ll']) + log_hastings):
        p['tail_mh_accept'][j,k]+=1
        p['tail_mh_type_accept'][move_type]+=1
        p['th'][j]=theta
        p['z']=base+theta[C[j]]
        p['ll']=ll
        return True
    sj['eta'][k]=old_eta
    return False

def tail_mh_sweep(p,train,omega,beta,i):
    """A deterministic, state-independent scan of vulnerable ratio coordinates."""
    # Two important near-zero dimensions receive independent proposals every sweep.
    for j,k in TAIL_PRIORITY:
        tail_beta_mh_coordinate(p,train,omega,beta,j,k)
    if i%TAIL_PRIOR_MH_EVERY==0:
        for j,coords in TAIL_FOCUS.items():
            # Coordinate scan rotates deterministically, independent of current state.
            k=coords[(i//TAIL_PRIOR_MH_EVERY-1)%len(coords)]
            if (j,k) not in TAIL_PRIORITY:
                tail_beta_mh_coordinate(p,train,omega,beta,j,k)
    # sampler_step reconstructs exact neural predictions immediately afterward.

# ---- Independent parallel-tempering ladders; exact cold target at beta=1 ----
PT_STATE_FIELDS=('s','th','g','mu','z','yh','ll','h')
def make_ladder(ref,train,seed):
    betas=np.geomspace(1.,PT_MIN_BETA,PT_TEMPS)
    replicas=[start_chain(ref,train,seed+1009*k,spread=(1.+1.5*k/max(1,PT_TEMPS-1)))
              for k in range(PT_TEMPS)]
    return dict(it=0,betas=betas,log_gaps=-np.diff(np.log(betas)),replicas=replicas,
                tries=np.zeros(PT_TEMPS-1,dtype=np.int64),
                accepts=np.zeros(PT_TEMPS-1,dtype=np.int64),
                window_tries=np.zeros(PT_TEMPS-1,dtype=np.int64),
                window_accepts=np.zeros(PT_TEMPS-1,dtype=np.int64),
                labels=np.arange(PT_TEMPS,dtype=np.int64),
                seen_hot=np.zeros(PT_TEMPS,dtype=bool),roundtrips=0,
                cold_ll_initial=float(replicas[0]['ll']))

def pt_exchange(ladder,rng):
    # Alternating even/odd adjacent swaps. Uses whole posterior state, NOT RNG/traces.
    parity=(ladder['it']//PT_SWAP_EVERY)%2
    for k in range(parity,PT_TEMPS-1,2):
        r1=ladder['replicas'][k];r2=ladder['replicas'][k+1]
        b1=float(ladder['betas'][k]);b2=float(ladder['betas'][k+1])
        log_accept=(b1-b2)*(float(r2['ll'])-float(r1['ll']))
        ladder['tries'][k]+=1;ladder['window_tries'][k]+=1
        if np.log(rng.random())<min(0.,log_accept):
            for key in PT_STATE_FIELDS:
                r1[key],r2[key]=r2[key],r1[key]
            ladder['labels'][k],ladder['labels'][k+1]=ladder['labels'][k+1],ladder['labels'][k]
            ladder['accepts'][k]+=1;ladder['window_accepts'][k]+=1
    ladder['seen_hot'][ladder['labels'][-1]]=True
    cold_label=int(ladder['labels'][0])
    if ladder['seen_hot'][cold_label]:
        ladder['roundtrips']+=1
        ladder['seen_hot'][cold_label]=False

def pt_adapt(ladder):
    # Adapt only during burn in; cumulative beta endpoints fixed.
    trials=ladder['window_tries'];acc=ladder['window_accepts']
    frac=np.divide(acc,np.maximum(1,trials))
    eligible=trials>0
    logg=np.log(ladder['log_gaps'])
    if eligible.any():
        rate=0.5/np.sqrt(1.+ladder['it']/float(PT_ADAPT_EVERY))
        logg[eligible]+=rate*(frac[eligible]-PT_TARGET_SWAP)
        vals=np.exp(logg-np.max(logg));gaps=vals/vals.sum()*(-np.log(PT_MIN_BETA))
        ladder['log_gaps']=gaps
        ladder['betas']=np.r_[1.,np.exp(-np.cumsum(gaps))]
    ladder['window_tries'][:]=0;ladder['window_accepts'][:]=0

def pt_one_sweep(ladder,train,omega):
    i=ladder['it']+1
    for k,rep in enumerate(ladder['replicas']):
        if k>0 and i%PT_HOT_SWEEP_EVERY:continue
        # Rotate disjoint 4-predictor blocks across successive hot sweeps;
        # each block update leaves the *full* tempered posterior invariant.
        if k==0:
            subset=None
        else:
            m=CFG['m']
            st=((i//PT_HOT_SWEEP_EVERY-1)*PT_HOT_PREDICTORS+(k%3)*PT_HOT_PREDICTORS)%m
            subset=tuple((st+r)%m for r in range(PT_HOT_PREDICTORS))
        sampler_step(rep,train,omega,beta=float(ladder['betas'][k]),record=(k==0),
                     global_move=(i%PT_GLOBAL_MOVE_EVERY==0),
                     prior_refresh=(i%PT_PRIOR_REFRESH_EVERY==0),
                     spectral_block=(k==0 and i%PT_SPECTRAL_BLOCK_EVERY==0),
                     coupled_tail=(k==0 and i%PT_COUPLED_TAIL_EVERY==0),
                     predictor_subset=subset,
                     targeted_tail_mh=(k==0))
    ladder['it']=i
    # A lightweight cache-consistency audit detects invalid state exchanges.
    if i in (1, 10) or i%1000==0:
        C,U,Y=train
        for rep in ladder['replicas']:
            check_z=rep['mu']+U@rep['g']
            for j in range(CFG['m']): check_z+=rep['th'][j][C[j]]
            fresh_ll=loglik(Y,phi(check_z,omega))
            if not np.allclose(rep['ll'],fresh_ll,rtol=0,atol=1e-7):
                raise AssertionError('Parallel tempering cached log-likelihood mismatch')
    if i%PT_SWAP_EVERY==0:
        pt_exchange(ladder,ladder['replicas'][0]['rng'])
    if i==1000:
        rates=ladder['accepts']/np.maximum(1,ladder['tries'])
        if np.min(rates)<PT_MIN_SWAP_WARN:
            print('EARLY PT ALERT: weakest adjacent swap rate',round(float(np.min(rates)),3),
                  'after 1000 iterations. Dense ladder may still need retuning.',flush=True)
    if i<=MCMC_BURN and i%PT_ADAPT_EVERY==0:
        pt_adapt(ladder)

def pt_report(ladders):
    print('PARALLEL TEMPERING (replica exchanges; cold posterior remains beta=1):',flush=True)
    rows=[]
    for ci,L in enumerate(ladders):
        for k in range(PT_TEMPS-1):
            tries=int(L['tries'][k]);ok=int(L['accepts'][k])
            rows.append(dict(chain=ci+1,pair=f'{k+1}-{k+2}',
                             beta1=float(L['betas'][k]),beta2=float(L['betas'][k+1]),
                             accepted=ok,attempts=tries,rate=(ok/tries if tries else np.nan)))
    table=pd.DataFrame(rows)
    print(table.round(4).to_string(index=False),flush=True)
    print('Roundtrips (hot->cold):',[int(L['roundtrips']) for L in ladders],flush=True)
    rate=table['rate'].to_numpy()
    bottleneck=~np.isfinite(rate)|(rate<PT_MIN_SWAP_WARN)
    if bottleneck.any():
        print('WARNING: parallel-tempering swap bottleneck (below 8% or undefined).',flush=True)
    print('Cold-chain JOINT (Gamma,mu) ESS evals/update:',
          [round(L['replicas'][0]['joint_effort']/max(1,L['replicas'][0]['joint_updates']),2) for L in ladders],flush=True)
    print('Per-predictor conditional-prior block refresh acceptance (cold only):',
          np.round(sum((L['replicas'][0]['refresh_accept'] for L in ladders)) /
                   np.maximum(1,sum((L['replicas'][0]['refresh_try'] for L in ladders))),3).tolist(),flush=True)
    print('Cold-chain GLOBAL Gaussian-block ESS evals/update:',
          [round(L['replicas'][0]['global_effort']/max(1,L['replicas'][0]['global_updates']),2) for L in ladders],flush=True)
    be=sum((L['replicas'][0]['spectral_block_evals'] for L in ladders));bn=sum((L['replicas'][0]['spectral_block_updates'] for L in ladders))
    ce=sum((L['replicas'][0]['coupled_tail_evals'] for L in ladders));cn=sum((L['replicas'][0]['coupled_tail_updates'] for L in ladders))
    print('Cold joint Beta-spectral ESS evaluations/update:',np.round(be/np.maximum(bn,1),2).tolist(),flush=True)
    print('Cold coupled-tail ESS evaluations/update:',np.round(ce/np.maximum(cn,1),2).tolist(),flush=True)
    tr=sum((L['replicas'][0]['tail_mh_try'] for L in ladders))
    ac=sum((L['replicas'][0]['tail_mh_accept'] for L in ladders))
    print('TARGETED TAIL multiscale Beta-prior / logit-RW MH acceptance (j,ratio_index,accepted/tried):',flush=True)
    for j in sorted(TAIL_FOCUS):
        for k in TAIL_FOCUS[j]:
            if tr[j,k]:print(f' j={j+1}, ratio={k+1}: {ac[j,k]}/{tr[j,k]} = {ac[j,k]/tr[j,k]:.3f}',flush=True)
    by_type_tr=sum((L['replicas'][0]['tail_mh_type_try'] for L in ladders))
    by_type_ac=sum((L['replicas'][0]['tail_mh_type_accept'] for L in ladders))
    print('MH proposal acceptance by type [multiscale independence, logit random walk]:',
          np.round(by_type_ac/np.maximum(by_type_tr,1),3).tolist(),flush=True)
    # Bottlenecks inform troubleshooting, but only target-posterior Rhat/ESS are mandatory convergence gates.
    return table

# ---- Honest convergence checks; invariants (spectral tails) are primary diagnostic targets ----
def diag_one(arr):
    ar=np.asarray(arr,dtype=float)
    if ar.ndim!=2 or ar.shape[0]<2 or ar.shape[1]<8:return (np.nan,np.nan,np.nan,np.nan)
    if not np.isfinite(ar).all():return (np.nan,np.nan,np.nan,np.nan)
    if any(np.ptp(ar[i])<1e-12 for i in range(ar.shape[0])):return (np.nan,np.nan,np.nan,np.nan)
    try:
        return (float(az.rhat(ar,method='rank')),
                float(az.ess(ar,method='bulk')),
                float(az.ess(ar,method='tail',prob=(.05,.95))),
                float(np.asarray(az.mcse(ar,method='mean')).item()))
    except Exception:
        return (np.nan,np.nan,np.nan,np.nan)

def diagnostics(chains, label, target_iters):
    c=CFG;K=c['klist'];m=c['m']
    s=np.stack([np.asarray(p['traces']['s2']) for p in chains])
    h=np.stack([np.asarray(p['traces']['bar']) for p in chains]);h0=np.stack([p['traces']['bar0'] for p in chains]);kap=np.stack([p['traces']['kappa'] for p in chains])
    ll=np.stack([p['traces']['ll'] for p in chains]);sig=np.stack([p['traces']['sigma2'] for p in chains])
    g=np.stack([np.asarray(p['traces']['gamma']) for p in chains]);mu=np.stack([np.asarray(p['traces']['mu']) for p in chains])
    rows=[]
    def add(name,v,core=True):
        rh,eb,et,mc=diag_one(v)
        rows.append(dict(quantity=name,rhat=rh,ess_bulk=eb,ess_tail=et,mcse=mc,mean=float(np.mean(v)),sd=float(np.std(v)),core=core))
    for j in range(m):
        add(f'j{j+1}:bar_rho',h[:,:,j])
        for k in range(K[j]):add(f'j{j+1}:s2_{k+1}',s[:,:,j,k])
        for d in range(1,K[j]):add(f'j{j+1}:tail_{d}',np.sum(s[:,:,j,d:K[j]],axis=-1))
        # This is NOT a posterior rank probability; derived effective-rank diagnostic only.
    add('hyper:bar0',h0);add('hyper:kappa',kap);add('sigma2',sig);add('log_collapsed_like',ll)
    for ii in range(c['pu']):
        for kk in range(c['pz']):add(f'gamma_{ii+1}_{kk+1}',g[:,:,ii,kk])
    for kk in range(c['pz']):add(f'mu_{kk+1}',mu[:,:,kk])
    df=pd.DataFrame(rows)
    okmask=df['core']
    fail=okmask & ((~np.isfinite(df.rhat)) | (df.rhat>RHAT_LIMIT) |
                   (~np.isfinite(df.ess_bulk)) | (df.ess_bulk<ESS_BULK_MIN) |
                   (~np.isfinite(df.ess_tail)) | (df.ess_tail<ESS_TAIL_MIN))
    print('\n'+'='*80,flush=True)
    print('MCMC DIAGNOSTICS:',label,'| per chain iters=',target_iters,'| retained=',s.shape[1], ' | chains=',len(chains),flush=True)
    print('Rhat and ESS are split-rank ArviZ estimates. Undefined values are not treated as successes.')
    with pd.option_context('display.max_rows',None,'display.width',170):
        print(df.round(4).to_string(index=False),flush=True)
    print(f"ALL-PARAMETER GATE: {'PASS' if not fail.any() else 'FAIL/INCONCLUSIVE'} | failing {int(fail.sum())}/{int(okmask.sum())}",flush=True)
    print('FAILING PARAMETERS (worst first):',flush=True)
    if fail.any():print(df.loc[fail].sort_values(['ess_bulk','rhat']).round(4).to_string(index=False),flush=True)
    else:print(' none',flush=True)
    relevant=df.quantity.str.contains(':tail_|:s2_|log_collapsed_like|sigma2|hyper:|:bar_rho')
    relfail=fail & relevant
    print(f"ALLOCATION-RELEVANT GATE: {'PASS' if not relfail.any() else 'FAIL'} | failing {int(relfail.sum())}/{int(relevant.sum())}",flush=True)
    print('Median bulk ESS (descriptive, not a replacement for the minimum):',round(float(df.loc[okmask,'ess_bulk'].median()),1),flush=True)
    # Decision-specific stability is distinct from marginal chain convergence.
    # It is reported for diagnosis only and NEVER substitutes for the strict gate.
    print('ALLOCATION STABILITY ACROSS COLD CHAINS (descriptive; cannot override failed Rhat/ESS):',flush=True)
    try:
        for xi in (0.,1.):
            pooled=dp_allocate(posterior_scores(s,xi),c['budgets'])
            per_chain=[dp_allocate(posterior_scores(s[cc:cc+1],xi),c['budgets']) for cc in range(s.shape[0])]
            for B in c['budgets']:
                d=pooled[B]['d']
                choices=[per_chain[k][B]['d'] for k in range(s.shape[0])]
                print(f'  xi={xi:g} budget={B}: pooled={d}, agreement={sum(t==d for t in choices)}/{len(choices)}',flush=True)
    except Exception as exc:
        print('  Allocation stability unavailable:',repr(exc),flush=True)

    print(f"Core max Rhat={df.loc[okmask,'rhat'].max():.4f}, min bulk ESS={df.loc[okmask,'ess_bulk'].min():.1f}, min tail ESS={df.loc[okmask,'ess_tail'].min():.1f}",flush=True)
    if (df.loc[okmask,'rhat']>1.01).any():print('WARNING: stricter publication-standard Rhat<=1.01 not met for all core quantities.',flush=True)
    print('CHAIN-LEVEL DRIFT CHECKS (early retained mean -> late retained mean):',flush=True)
    for ci in range(len(chains)):
        mid=s.shape[1]//2
        if mid:
            lo=float(ll[ci,:mid].mean());hi=float(ll[ci,mid:].mean())
            x=float(np.mean(s[ci,:mid,0,:K[0]].sum(axis=-1)))
            y=float(np.mean(s[ci,mid:,0,:K[0]].sum(axis=-1)))
            print(f' chain={ci+1} logLik_mean={lo:.3f}->{hi:.3f}; predictor_1_total_s2={x:.5f}->{y:.5f}; kappa={kap[ci].mean():.3f}',flush=True)
    print('SLICE EFFORT: likelihood evaluations per update (not MH acceptance rates):',flush=True)
    effort=sum((p['effort'] for p in chains));updates=sum((p['updates'] for p in chains))
    print(pd.DataFrame({'predictor':np.arange(1,m+1),
                        'orient_U_calls':np.round(effort[:,0]/np.maximum(1,updates[:,0]),2),
                        'orient_V_calls':np.round(effort[:,1]/np.maximum(1,updates[:,1]),2),
                        'spectrum_calls':np.round(effort[:,2]/np.maximum(1,updates[:,2]),2)}).to_string(index=False),flush=True)
    he=sum((p['h_effort'] for p in chains));hn=sum((p['h_updates'] for p in chains))
    print('Hyper slice evaluations per update [bar0, bar_j, kappa]:',np.round(he/np.maximum(hn,1),2).tolist(),flush=True)
    print('IMPLIED EFFECTIVE DIMENSIONS (99% of spectral energy) — descriptive, NOT sampled exact ranks:',flush=True)
    for j in range(m):
        v=s[:,:,j,:K[j]].reshape(-1,K[j]);cumulative=np.cumsum(v,axis=-1)/np.maximum(1e-300,v.sum(-1,keepdims=True))
        ranks=1+np.argmax(cumulative>=.99,axis=1)
        print(f' j={j+1}: '+str({int(k):round(float(np.mean(ranks==k)),3) for k in range(1,K[j]+1)}),flush=True)
    print('='*80,flush=True)
    return (bool(not fail.any()),df,s)

# ---- Exact multiple-choice knapsack; predictor-specific posterior mean + xi*sd ----
def dp_allocate(scores,budgets):
    c=CFG;Bmax=max(budgets);m=c['m'];N=c['nlev'];K=c['klist']
    F=np.full((m+1,Bmax+1),np.inf);F[0,0]=0
    back=np.full((m+1,Bmax+1),-1,int)
    for j in range(m):
        for b in range(Bmax+1):
            if not np.isfinite(F[j,b]):continue
            for d in range(1,K[j]+1):
                b2=b+N[j]*d
                if b2<=Bmax and F[j,b]+scores[j][d-1]<F[j+1,b2]:
                    F[j+1,b2]=F[j,b]+scores[j][d-1];back[j+1,b2]=d
    out={}
    for B in budgets:
        b=int(np.argmin(F[m,:B+1]));ds=[0]*m;cost=b
        if not np.isfinite(F[m,b]):raise RuntimeError('No feasible allocation')
        for j in range(m,0,-1):
            d=int(back[j,b]);ds[j-1]=d;b-=N[j-1]*d
        out[B]=dict(d=tuple(ds),cost=cost,score=float(F[m,cost]))
    return out

def posterior_scores(s,xi):
    c=CFG;out=[]
    for j in range(c['m']):
        vals=s[:,:,j,:c['klist'][j]].reshape(-1,c['klist'][j])
        arr=[]
        for d in range(1,c['klist'][j]+1):
            tail=vals[:,d:].sum(1)
            arr.append(float(np.mean(tail)+xi*np.std(tail,ddof=1)))
        out.append(arr)
    return out

def score_discrepancy(mean_scores,oracle_scores):
    # Exact sup |sum_j (S_j(d_j)-L0_j(d_j))| over the Cartesian dimension set.
    # It does not require enumerating exponentially many allocations.
    deviations=[np.asarray(mean_scores[j])-np.asarray(oracle_scores[j]) for j in range(CFG['m'])]
    delta=max(abs(sum(float(x.max()) for x in deviations)),
              abs(sum(float(x.min()) for x in deviations)))
    base=sum(float(oracle_scores[j][0]) for j in range(CFG['m']))
    return (float(delta),float(delta/max(base,1e-12)))

def truth_scores(truth):
    return [[float(np.sum(np.asarray(truth['spec'][j][d:])**2)) for d in range(1,CFG['klist'][j]+1)] for j in range(CFG['m'])]

def print_posterior_recovery(s,truth):
    c=CFG
    print('PER-PREDICTOR POSTERIOR SPECTRUM vs GENERATING SPECTRUM:',flush=True)
    for j in range(c['m']):
        vals=s[:,:,j,:c['klist'][j]].reshape(-1,c['klist'][j])
        actual=np.asarray(truth['spec'][j])**2
        mean=vals.mean(0);lo,hi=np.quantile(vals,[.025,.975],axis=0)
        for k in range(c['klist'][j]):
            print(f'  predictor={j+1:02d} eigen={k+1} TRUE={actual[k]:.5f} POST_MEAN={mean[k]:.5f} CI95=({lo[k]:.5f},{hi[k]:.5f})',flush=True)
        for d in range(1,c['klist'][j]):
            v=vals[:,d:].sum(1);t=float(actual[d:].sum())
            print(f'  predictor={j+1:02d} tail(d={d}) TRUE={t:.5f} MEAN={np.mean(v):.5f} SD={np.std(v,ddof=1):.5f} CI95=({np.quantile(v,.025):.5f},{np.quantile(v,.975):.5f})',flush=True)

# ---- Checkpoint-aware end-to-end replicate: no synthetic convergence claims ----
def run_one(task):
    global CFG
    sim,regime,ntrain,rep=task
    CFG=setup(sim)
    c=CFG
    name=f'Simulation{sim}_{regime}_n{ntrain}_rep{rep:03d}'
    key=str(task)
    STATE['task_name']=name
    w=STATE['work']
    if w.get('task')!=task:
        seed=(271828 if sim=='1' else 314159)+100003*rep+1009*c['scenarios'].index((regime if sim=='2' else f'{regime}_n{ntrain}',regime,ntrain))
        # In Simulation 1, the scenario's regime is clear/weak; in Simulation 2 the same as name.
        truth=target(regime)
        t=teacher(sim)
        w=dict(task=task,name=name,phase='reference',seed=seed,truth=truth,teacher=t,
               training=data(ntrain,truth,t,seed+1),validation=data(c['n_val'],truth,t,seed+2),
               test=data(c['n_test'],truth,t,seed+3))
        STATE['work']=w
        atomic_checkpoint(force=True)
    print(f'\n>>> {name} | phase={w["phase"]}',flush=True)
    if w['phase']=='reference':
        net,loss=fit_model(w['training'],'reference',w['seed']+10)
        w['ref_params']=reference_parameters(net)
        w['reference_train_mse']=loss
        w['reference_test_mse']=mse(w['test'][2],predict(net,w['test']))
        del net;gc.collect()
        w['phase']='mcmc';atomic_checkpoint(force=True)
    if w['phase']=='mcmc':
        configure_fast_likelihood(w['training'],w['ref_params']['omega'])
        if 'ladders' not in w:
            w['ladders']=[make_ladder(w['ref_params'],w['training'],w['seed']+10000+21111*i) for i in range(N_CHAINS)]
            w['target_iter']=MCMC_BASE
            atomic_checkpoint(force=True)
        ladders=w['ladders']
        while True:
            for i,L in enumerate(ladders):
                if L['it']<w['target_iter']:
                    print(f'  independent PT ladder {i+1}/{N_CHAINS}: {L["it"]} -> {w["target_iter"]}',flush=True)
                while L['it']<w['target_iter']:
                    pt_one_sweep(L,w['training'],w['ref_params']['omega'])
                    if L['it']%100==0:
                        if L['it']%500==0:
                            print(f'   PT ladder {i+1} iter={L["it"]} cold_logLik={L["replicas"][0]["ll"]:.2f} '
                                  f'roundtrips={L["roundtrips"]}',flush=True)
                        atomic_checkpoint()
                atomic_checkpoint(force=True)
            chains=[L['replicas'][0] for L in ladders]
            pt_report(ladders)
            passed,diagnostics_df,s=diagnostics(chains,name,w['target_iter'])
            if passed:
                w['passed']=True;break
            if w['target_iter']>=MCMC_MAX:
                w['passed']=False;break
            w['target_iter']=min(MCMC_MAX,w['target_iter']+MCMC_EXTEND)
            print('EXTENDING PARALLEL-TEMPERING LADDERS for failed/inconclusive diagnostics to',w['target_iter'],flush=True)
            atomic_checkpoint(force=True)
        w['posterior_s2']=s
        w['posterior_summary']=diagnostics_df.to_dict('records')
        if not w['passed']:
            print('BAYESIAN CONVERGENCE FAILED/INCONCLUSIVE. Detailed printed diagnostics above.',flush=True)
            if REQUIRE_CONVERGENCE:
                w['phase']='failed';atomic_checkpoint(force=True)
                return dict(status='MCMC_FAILED',task=name,iterations=w['target_iter'])
        # retain only posterior draws; no large MCMC state needed after accepted checks
        h0=np.stack([p['traces']['bar0'] for p in chains]);bar=np.stack([p['traces']['bar'] for p in chains]);kap=np.stack([p['traces']['kappa'] for p in chains])
        print('HIERARCHICAL POSTERIOR MEANS: hypermean=',float(h0.mean()),'kappa=',float(kap.mean()),flush=True)
        print('per predictor mean ratios=',np.round(bar.mean((0,1)),4),flush=True)
        print_posterior_recovery(s,w['truth'])
        w.pop('ladders',None)
        alloc={};tgt=dp_allocate(truth_scores(w['truth']),c['budgets'])
        delta,relative_delta=score_discrepancy(posterior_scores(s,0),truth_scores(w['truth']))
        w['spectral_delta']=delta;w['normalized_spectral_delta']=relative_delta
        squared_errors=[];tail_errors=[];tail_coverage=[]
        for j in range(c['m']):
            vals=s[:,:,j,:c['klist'][j]].reshape(-1,c['klist'][j]);actual=np.asarray(w['truth']['spec'][j])**2
            squared_errors.extend((vals.mean(0)-actual)**2)
            for d in range(1,c['klist'][j]):
                tt=vals[:,d:].sum(1)
                tail_errors.append((tt.mean()-actual[d:].sum())**2)
                tail_coverage.append(float(np.quantile(tt,.025)<=actual[d:].sum()<=np.quantile(tt,.975)))
        w['posterior_s2_rmse']=float(np.sqrt(np.mean(squared_errors)))
        w['posterior_tail_rmse']=float(np.sqrt(np.mean(tail_errors)))
        w['posterior_tail_coverage']=float(np.mean(tail_coverage))
        print('POSTERIOR RECOVERY: s2 RMSE=',w['posterior_s2_rmse'],
              'tail RMSE=',w['posterior_tail_rmse'], '95% tail coverage=',w['posterior_tail_coverage'],
              'score sup discrepancy=',delta,'normalized=',relative_delta,flush=True)
        for xi in RISK_XI:
            aa=dp_allocate(posterior_scores(s,xi),c['budgets'])
            alloc[xi]=aa
            print(f'ALLOCATIONS xi={xi}:',flush=True)
            for B in c['budgets']:
                print(' budget=',B,' posterior=',aa[B]['d'],'score=',round(aa[B]['score'],5),
                      'true spectral oracle=',tgt[B]['d'],flush=True)
        w['alloc']=alloc;w['oracle']=tgt
        w['refits']={};w['phase']='refit';atomic_checkpoint(force=True)
    if w['phase']=='failed':return dict(status='MCMC_FAILED',task=name,iterations=w.get('target_iter'))
    if w['phase']=='refit':
        wanted=sorted(set(tuple(v['d']) for aa in w['alloc'].values() for v in aa.values()))
        print('REFITTING',len(wanted),'UNIQUE ALLOCATIONS across risk coefficients/budgets',flush=True)
        for d in wanted:
            if d in w['refits']:continue
            net,train_loss=fit_model(w['training'],'reduced',w['seed']+2000+sum((i+1)*z for i,z in enumerate(d)),d=d)
            v=mse(w['validation'][2],predict(net,w['validation']))
            te=mse(w['test'][2],predict(net,w['test']))
            w['refits'][d]=dict(train=train_loss,val=v,test=te)
            print('  REFIT',d,'cost=',sum(c['nlev'][j]*d[j] for j in range(c['m'])),
                  'VAL=',round(v,6),'TEST=',round(te,6),flush=True)
            del net;gc.collect();atomic_checkpoint(force=True)
        w['phase']='summary';atomic_checkpoint(force=True)
    if w['phase']=='summary':
        results=[];baseline=max(c['budgets'])
        for xi,aa in w['alloc'].items():
            largest=w['refits'][aa[baseline]['d']]['val']
            eligible=[B for B in c['budgets'] if w['refits'][aa[B]['d']]['val']<=largest+config['zeta']]
            Bsel=min(eligible)
            dsel=aa[Bsel]['d'];best=w['refits'][dsel]
            exact=np.mean([aa[B]['d']==w['oracle'][B]['d'] for B in c['budgets']])
            test_oracle_B=min(c['budgets'],key=lambda B:w['refits'][aa[B]['d']]['test'])
            out=dict(status='OK' if w['passed'] else 'MCMC_FAILED_WARNING',task=name,sim=sim,regime=regime,ntrain=ntrain,rep=rep,
                     xi=float(xi),Bsel=Bsel,dsel=str(dsel),cost=int(sum(c['nlev'][j]*dsel[j] for j in range(c['m']))),
                     val_mse=best['val'],test_mse=best['test'],baseline_max_test_mse=w['refits'][aa[baseline]['d']]['test'],
                     ref_test_mse=w['reference_test_mse'],oracle_budget_allocation_exact_rate=exact,
                     test_oracle_B=test_oracle_B, budget_match_with_test_oracle=int(Bsel==test_oracle_B),
                     test_oracle_mse=w['refits'][aa[test_oracle_B]['d']]['test'],
                     test_excess_vs_max_budget=best['test']-w['refits'][aa[baseline]['d']]['test'],
                     s2_rmse=w['posterior_s2_rmse'],tail_rmse=w['posterior_tail_rmse'],
                     tail_coverage=w['posterior_tail_coverage'],spectral_delta=w['spectral_delta'],
                     normalized_spectral_delta=w['normalized_spectral_delta'],
                     target_dim_accuracy=float(np.mean(np.asarray(dsel)==w['truth']['dtrue'])) if w['truth']['dtrue'] else np.nan,
                     mcmc_iters=int(w['target_iter']))
            results.append(out)
            print('\nSELECTED RESULT:',out,flush=True)
        w['results']=results
        w['phase']='done';atomic_checkpoint(force=True)
        return results
    if w['phase']=='done':return w['results']
    raise RuntimeError('Unexpected phase: '+str(w['phase']))

TASKS=[]
for sim in SIMULATIONS:
    c=setup(sim)
    for label,regime,n in c['scenarios']:
        for rep in range(1,c['num_reps']+1):TASKS.append((sim,regime,n,rep))
print('\nCONFIG:',json.dumps(config,indent=2),'\nTOTAL REPLICATION TASKS:',len(TASKS),'\n',flush=True)
try:
    for itask,task in enumerate(TASKS,1):
        k=str(task)
        if k in STATE['completed']:continue
        print(f'\n[{itask}/{len(TASKS)}] START TASK {task}',flush=True)
        res=run_one(task)
        STATE['completed'][k]=res if isinstance(res,list) else [res]
        # Preserve the failing chains, traces, and reference fit rather than erasing
        # every diagnostic draw when a convergence failure occurs.
        if not any(r.get('status')=='MCMC_FAILED' for r in STATE['completed'][k]):
            STATE['work']={}
        atomic_checkpoint(force=True)
        if STOP_ON_FIRST_FAILURE and any(r.get('status')=='MCMC_FAILED' for r in STATE['completed'][k]):
            print('STOPPED AFTER NONCONVERGENCE: inspect the printed diagnostics before rerunning or changing the model.',flush=True)
            break
except KeyboardInterrupt:
    print('\nUSER INTERRUPTION: saving last known valid checkpoint.',flush=True)
    atomic_checkpoint(force=True);raise
except Exception:
    print('\nERROR DETECTED (not suppressed). Last state checkpoint follows:',flush=True)
    traceback.print_exc();atomic_checkpoint(force=True);raise

ALL=[x for v in STATE['completed'].values() for x in v]
RESULT_DF=pd.DataFrame(ALL)
print('\n\nALL REPLICATION RESULTS\n',RESULT_DF.to_string(index=False),flush=True)
if not RESULT_DF.empty and 'status' in RESULT_DF and 'xi' in RESULT_DF:
    ok=RESULT_DF[RESULT_DF['status']=='OK']
    if len(ok):
        print('\nGROUPED SIMULATION RESULTS\n',ok.groupby(['sim','regime','ntrain','xi'],dropna=False)
            .agg(n=('rep','count'),mean_selected_B=('Bsel','mean'),mean_test_mse=('test_mse','mean'),
                 mean_reference_test_mse=('ref_test_mse','mean'),mean_allocation_exact=('oracle_budget_allocation_exact_rate','mean'),
                 mean_target_dim_accuracy=('target_dim_accuracy','mean'),
                 mean_tail_rmse=('tail_rmse','mean'),mean_spectral_delta=('spectral_delta','mean'),
                 mean_normalized_delta=('normalized_spectral_delta','mean'),
                 budget_test_oracle_match=('budget_match_with_test_oracle','mean'),
                 mean_test_excess=('test_excess_vs_max_budget','mean')).round(5).to_string(),flush=True)
print('\nCOMPLETED; all print-only MCMC diagnostics above. Checkpoint:',CPATH,flush=True)
print('A PASS is evidence from these diagnostics, not proof of exact posterior convergence. No rank probability is estimated.',flush=True)


NumPy BLAS threads limited to 1 for small-matrix MCMC computations.
Torch 2.11.0+cpu | device: cpu | sampler: cached likelihood/auto JIT + conditional hyperprior; neural nets: torch GPU if available
Mounted at /content/drive

RUN SELECTION | /content/drive/MyDrive/JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V6_TAIL_MH/Simulation2/diagnostic
[1] START FRESH (new simulated data and reference network)
[2] REUSE V5/V4 DATA+REFERENCE; RESTART all four V6 MCMC ladders
[3] RESUME V6 CHECKPOINT (same configuration)
Available fitted-reference imports: [('V4', '/content/drive/MyDrive/JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V4_FUSED/Simulation2/diagnostic/checkpoint.pkl')]
V6 RESUMED: Simulation2_heterogeneous_n1200_rep001 | phase: mcmc

CONFIG: {
  "version": "FixedRank_HierarchicalBeta_RiskSensitive_PT_SIM2_v6_targeted_beta_tail_mh_exact",
  "simulations": [
    "2"
  ],
  "full": false,
  "risk_xi": [
    0.0,
    0.5,
    1.0
  ],
  "n_cha

In [ ]:
# SIMULATION 2 — ONE SELF-CONTAINED GOOGLE COLAB CELL
# Fixed-maximum-rank multiplicative spectral shrinkage: NO RJMCMC and NO discrete rank sampler.
# V7 exact-target repair: fixed-scale tilted-Beta independence MH and joint orientation/tail moves,
# calibrated reference fitting, SVD-initialized predictive refits, cached/JIT likelihood.
# Hierarchical Beta spectral ratios and predictor-separable risk-sensitive allocation.
# Convergence is measured, not assumed. Exact beta-tempered, fixed-dimension posterior sweeps; minimum ESS gate unchanged.
# Checkpoints saved to Google Drive every 20 minutes and at phase boundaries.
# The new MCMC update schedule starts fresh chains. Optional V5/V4 import reuses data/reference fit ONLY;
# V7 checkpoints resume exactly; V6 checkpoints are strictly read-only.
# Main results and convergence tables are saved to CSV; converged posterior spectra to NPZ.
# All diagnostics also print to console, and checkpoint state is restartable.
# ===========================================================================
# SIMULATION 2: INDEPENDENT CODE AND CHECKPOINTS; no other notebook cells required.
# EDIT THESE BEFORE RUNNING A NEW EXPERIMENT:
SIMULATIONS = ('2',)      # Fixed for Simulation 2; independently runnable
RUN_FULL = False         # False: 1 replication per regime (diagnostic); True: paper-scale repetitions
REFERENCE_PREFLIGHT_ONLY = False  # True: fit/verify one reference, save checkpoint, then use option 3 with False
RISK_XI = (0.0, 0.5, 1.0)  # fixed, predeclared uncertainty penalties
REQUIRE_CONVERGENCE = True   # do not interpret/refit failed Bayesian chains
STOP_ON_FIRST_FAILURE = True # stop expensive replications so diagnostics can be inspected
# MCMC settings apply PER independent chain. Extensions are automatic until pass or limit.
N_CHAINS = 4
MCMC_BASE = 4500 if not RUN_FULL else 6000
MCMC_EXTEND = 1500
MCMC_MAX = 15000 if not RUN_FULL else 21000
MCMC_BURN = 2000 if not RUN_FULL else 3000
MCMC_THIN = 2 if not RUN_FULL else 4
RHAT_LIMIT = 1.01          # strict R-hat gate for ALL reported posterior parameters
ESS_BULK_MIN = 400
ESS_TAIL_MIN = 400
CHECKPOINT_SECONDS = 20 * 60
# Temperatures temper ONLY the collapsed likelihood; priors stay untempered.
PT_TEMPS = 12 if not RUN_FULL else 16
PT_MIN_BETA = 0.20 if not RUN_FULL else 0.12
PT_SWAP_EVERY = 2
PT_ADAPT_EVERY = 100   # temperature-ladder adaptation stops at burn-in
PT_TARGET_SWAP = 0.25
PT_MIN_SWAP_WARN = 0.08
PT_GLOBAL_MOVE_EVERY = 8  # occasional joint orientation and regression move
PT_PRIOR_REFRESH_EVERY = 10  # retain original exact independent conditional-prior ratio proposals
PT_HOT_SWEEP_EVERY = 2       # each hot replica updates every second sweep; cold updates every sweep
PT_HOT_PREDICTORS = 4         # cyclic subset of the 12 predictors per hot sweep; conditional kernels preserve tempered target
PT_SPECTRAL_BLOCK_EVERY = 6  # exact tilted-Beta joint vector independence MH (cold chain)
PT_COUPLED_TAIL_EVERY = 4    # exact joint auxiliary-orientation and Beta-ratio independence MH
# Targeted, exact MH moves for near-zero spectral ratios. The proposal depends
# ONLY on fixed scales and current hierarchical parameters; MH corrects for q.
# Unlike a fixed narrow truncation, all mixture components have full (0,1) support.
TAIL_PRIOR_MH_EVERY = 2
TAIL_TILT_LAMBDAS = (0.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0, 1000000.0)
TAIL_TILT_WEIGHTS = (0.12, 0.08, 0.16, 0.26, 0.20, 0.12, 0.06)
TAIL_RATIO_SLICE_WIDTH = 1.8
TAIL_FOCUS = {2:(1,2),3:(0,1),5:(1,2),7:(0,1,2,3),
              8:(1,),10:(0,),11:(2,3)}
TAIL_PRIORITY = ((7,0), (11,2))  # original difficult coordinates, zero based
TAIL_PRIORITY_REPEATS = 3         # additional exact kernel applications per cold sweep
REF_EPOCHS = 1800 if not RUN_FULL else 1800
REF_RESTARTS = 5
REDUCED_EPOCHS = 700
REDUCED_PATIENCE = 140
REDUCED_LR = 0.003
REDUCED_WEIGHT_DECAY = 5e-5
# All extra moves are valid invariant kernels. Hot subset steps reduce cost of the denser ladder;
# their mixing efficiency must still be verified, not assumed.
# Do not misinterpret a failed posterior as 'success' based on high median ESS.
# All parameters are reported; strict convergence gate is applied to all.

# ===========================================================================
import os, time, json, pickle, random, hashlib, gc, math, warnings, shutil, traceback, copy
from datetime import datetime
import numpy as np
import pandas as pd
from scipy.special import expit, log_expit, gammaln, betaln, logsumexp
import torch
from torch import nn
from torch.nn import functional as F
try:
    import arviz as az
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'arviz'])
    import arviz as az
warnings.filterwarnings('ignore', category=FutureWarning)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(max(1,min(4,os.cpu_count() or 1)))
# Tiny (5x5) NumPy/BLAS operations dominate the MCMC inner loop; avoid
# expensive multithread startup on every candidate. This changes no target.
try:
    from threadpoolctl import threadpool_limits
    _blas_guard = threadpool_limits(limits=1,user_api='blas')
    print('NumPy BLAS threads limited to 1 for small-matrix MCMC computations.',flush=True)
except ImportError:
    _blas_guard = None
print('Torch',torch.__version__,'| device:',DEVICE,'| sampler: cached likelihood/auto JIT + conditional hyperprior; neural nets: torch GPU if available',flush=True)
try:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except Exception as exc:
        raise RuntimeError('Google Drive mount failed; authorize access and rerun. Drive checkpoint is mandatory in Colab.') from exc
    ROOT = '/content/drive/MyDrive/JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V7_EXACT_TAIL_MIXING'
except ImportError:
    ROOT = os.path.abspath('./JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V7_EXACT_TAIL_MIXING')
    print('NOTE: outside Colab; checkpoint directory is LOCAL, not Google Drive.',ROOT)
RUN_DIR = os.path.join(ROOT, 'Simulation2', 'full' if RUN_FULL else 'diagnostic')
os.makedirs(RUN_DIR,exist_ok=True)
CPATH = os.path.join(RUN_DIR, 'checkpoint.pkl')
config = dict(version='FixedRank_HierarchicalBeta_RiskSensitive_PT_SIM2_v7_tilted_beta_SVD_anchor', simulations=SIMULATIONS, full=RUN_FULL,
              risk_xi=RISK_XI, n_chains=N_CHAINS, iters=(MCMC_BASE,MCMC_EXTEND,MCMC_MAX),
              burn=MCMC_BURN, thin=MCMC_THIN, rhat_limit=RHAT_LIMIT, ess=(ESS_BULK_MIN,ESS_TAIL_MIN),
              hyperprior=dict(a0=2.,b0=2.,pool=4.,ak=2.,bk=1.), s1_prior=(2.,1.),
              gamma_sd=2.5,mu_sd=2.5, sigma_ig=(2.5,0.02), zeta=0.0015,
              training=dict(ref_lr=.005,red_lr=REDUCED_LR,red_weight_decay=REDUCED_WEIGHT_DECAY),
              checkpoint_seconds=CHECKPOINT_SECONDS, stop_on_first_failure=STOP_ON_FIRST_FAILURE,
              pt=(PT_TEMPS,PT_MIN_BETA,PT_SWAP_EVERY,PT_ADAPT_EVERY,PT_GLOBAL_MOVE_EVERY,PT_PRIOR_REFRESH_EVERY),
              hot_sweep_every=PT_HOT_SWEEP_EVERY,hot_predictors=PT_HOT_PREDICTORS,
              spectral_block_every=PT_SPECTRAL_BLOCK_EVERY,coupled_tail_every=PT_COUPLED_TAIL_EVERY,
              tail_mh=(TAIL_PRIOR_MH_EVERY,TAIL_TILT_LAMBDAS,TAIL_TILT_WEIGHTS,
                         TAIL_RATIO_SLICE_WIDTH,tuple((j,tuple(inds)) for j,inds in TAIL_FOCUS.items()),
                         TAIL_PRIORITY,TAIL_PRIORITY_REPEATS),
              reference_optimizer=(REF_EPOCHS,REF_RESTARTS),
              reduced_optimizer=(REDUCED_EPOCHS,REDUCED_PATIENCE,REDUCED_LR,REDUCED_WEIGHT_DECAY))
SIGNATURE = hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()
# A different invariant MCMC transition schedule deserves a fresh, clearly labelled
# diagnostic run. Data, teacher and fitted reference parameters MAY be reused.
# This version changes the reference anchor and transition kernels. Accordingly,
# V6 traces are NEVER pooled with V7 traces, despite identical target-prior families.
_BASE_ROOT = ('/content/drive/MyDrive/JRSSB_Embedding_Allocation' if os.path.isdir('/content/drive/MyDrive')
              else os.path.abspath('./JRSSB_Embedding_Allocation'))
SOURCE_ROOTS = (
    ('V6',os.path.join(_BASE_ROOT,'FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V6_TAIL_MH')),
    ('V5',os.path.join(_BASE_ROOT,'FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V5_DENSE_TAIL')),
    ('V4',os.path.join(_BASE_ROOT,'FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V4_FUSED')),
)
REQUIRED_FIT = ('task','name','seed','truth','teacher','training','validation','test')
def recoverable_fit_sources():
    paths=[]
    for version,root in SOURCE_ROOTS:
        directory=os.path.join(root,'Simulation2','full' if RUN_FULL else 'diagnostic')
        trial=[os.path.join(directory,'checkpoint.pkl')]
        if os.path.isdir(directory):
            for entry in sorted(os.listdir(directory),reverse=True):
                arc=os.path.join(directory,entry,'checkpoint.pkl')
                if entry.startswith('archive_') and os.path.isfile(arc): trial.append(arc)
        for path in trial:
            if os.path.isfile(path):
                try:
                    with open(path,'rb') as f: candidate=pickle.load(f)
                    work=candidate.get('work',{})
                    if all(q in work for q in REQUIRED_FIT) and str(work['task'][0])=='2':
                        paths.append((version,path,work))
                except Exception as exc:
                    print('Unable to examine legacy checkpoint',path,repr(exc),flush=True)
    return paths
_FIT_SOURCES=recoverable_fit_sources()
print('\nRUN SELECTION |',RUN_DIR)
print('[1] START FRESH V7 (new data, calibrated reference, fresh posterior)')
print('[2] REUSE V6/V5/V4 DATA ONLY, RETRAIN calibrated reference, START new V7 posterior')
print('[3] RESUME V7 CHECKPOINT (including after reference-only preflight)')
if _FIT_SOURCES:
    print('Available data imports (old checkpoints read-only):',[(v,p) for v,p,_ in _FIT_SOURCES],flush=True)
else:
    print('No recoverable older datasets. Option 1 regenerates data deterministically.')
while True:
    MODE = input('Enter 1, 2 or 3: ').strip()
    if MODE=='1' or (MODE=='2' and _FIT_SOURCES) or (MODE=='3' and os.path.isfile(CPATH)):
        break
    print('Option 2 needs legacy simulated data; option 3 needs an existing V7 checkpoint.')
if MODE in ('1','2') and os.path.exists(CPATH):
    archived=os.path.join(RUN_DIR,'archive_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
    os.makedirs(archived,exist_ok=True)
    shutil.move(CPATH,os.path.join(archived,'checkpoint.pkl'))
    print('Previous V7 checkpoint archived:',archived)
if MODE=='3':
    with open(CPATH,'rb') as f: STATE=pickle.load(f)
    if STATE.get('signature')!=SIGNATURE:
        raise ValueError('V7 checkpoint signature mismatch: resume with the same settings or START FRESH.')
    print('V7 RESUMED:',STATE.get('task_name'),'| phase:',STATE.get('work',{}).get('phase'))
elif MODE=='2':
    v,path,oldw=_FIT_SOURCES[0]
    freshw={key:oldw[key] for key in REQUIRED_FIT}
    freshw['phase']='reference'
    STATE=dict(signature=SIGNATURE,completed={},work=freshw,created=datetime.now().isoformat(),
               task_name=freshw['name'])
    print(f'REUSED {v} DATA FROM {path}. Old reference and MCMC discarded; V7 anchor retrained.',flush=True)
else:
    STATE=dict(signature=SIGNATURE,completed={},work={},created=datetime.now().isoformat(),task_name=None)
LAST_SAVE = time.monotonic()

def atomic_checkpoint(force=False):
    global LAST_SAVE
    if not force and (time.monotonic()-LAST_SAVE)<CHECKPOINT_SECONDS:return
    tmp=CPATH+'.tmp'
    with open(tmp,'wb') as f:
        pickle.dump(STATE,f,protocol=pickle.HIGHEST_PROTOCOL)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp,CPATH)
    LAST_SAVE=time.monotonic()
    print('  [DRIVE CHECKPOINT]',datetime.now().strftime('%H:%M:%S'), '| phase:',STATE.get('work',{}).get('phase'), flush=True)

if MODE=='2':
    atomic_checkpoint(force=True)  # new V7 state; all V6/V5/V4 checkpoints untouched

def seed_all(seed):
    np.random.seed(seed);random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)

def orthogonal(k,rng):
    q,r=np.linalg.qr(rng.normal(size=(k,k)))
    return q * np.where(np.diag(r)>=0,1.,-1.)

def centered_orth(n,k,rng):
    a=rng.normal(size=(n,k));a-=a.mean(axis=0,keepdims=True)
    return np.linalg.qr(a)[0][:,:k]

def make_theta(n,pz,singular,rng):
    k=min(n-1,pz)
    t=centered_orth(n,k,rng) @ np.diag(np.asarray(singular[:k])) @ orthogonal(pz,rng)[:,:k].T
    return t-t.mean(axis=0,keepdims=True)

def setup(sim):
    if sim=='1':
        nlev=[8,10,12,9];pz=4;q=4;pu=2
        scen=[(f'{regime}_n{n}',regime,n) for regime in ('clear','weak') for n in (500,1000,2000)]
        if not RUN_FULL:scen=[x for x in scen if x[2]==1000]
        budgets=[sum(nlev),60,80,100,120,140,pz*sum(nlev)];num_reps=100 if RUN_FULL else 1
    else:
        nlev=[3,5,8,4,10,6,7,12,5,9,4,11];pz=5;q=5;pu=3
        scen=[(r,r,1200) for r in ('heterogeneous','near_homogeneous')]
        budgets=[sum(nlev),125,170,215,255,300,340,sum(x*min(x-1,pz) for x in nlev)]
        num_reps=50 if RUN_FULL else 1
    klist=[min(n-1,pz) for n in nlev]
    bases=[]
    for n in nlev:
        _,_,vt=np.linalg.svd(np.ones((1,n)),full_matrices=True)
        bases.append(vt[1:,:].T.copy())
    return dict(sim=sim,nlev=nlev,m=len(nlev),pz=pz,q=q,pu=pu,klist=klist,bases=bases,
                scenarios=scen,budgets=sorted(set(budgets)),num_reps=num_reps,
                n_val=500,n_test=(3000 if sim=='1' else 4000), noise_sd=.1,
                ref_epochs=REF_EPOCHS,red_epochs=REDUCED_EPOCHS,
                ref_restarts=REF_RESTARTS,red_restarts=1)

CFG=None

def teacher(sim):
    rng=np.random.default_rng(8877 if sim=='1' else 90210)
    p=CFG['pz']
    if sim=='1':b1=np.array([.1,-.1,.05,-.05]);b2=np.array([.05,-.05,.05,-.05])
    else:b1=np.array([.1,-.05,.05,-.1,.05]);b2=np.array([.05,-.05,.05,0,-.05])
    return [(orthogonal(p,rng),b1),(orthogonal(p,rng),b2)]

def target(regime):
    c=CFG; sim=c['sim'];m=c['m'];nlev=c['nlev'];klist=c['klist']
    rng=np.random.default_rng(12345 if sim=='1' else 4567)
    if sim=='1':
        raw=([[1.8,.3,.1,.05],[1.6,.7,.2,.05],[2.,1.,.4,.1],[1.5,.5,.2,.05]] if regime=='clear'
             else [[1.8,1.2,.9,.6],[1.6,1.3,1.,.7],[2.,1.6,1.3,1.],[1.5,1.1,.85,.6]])
        spec=[np.array(raw[j][:klist[j]]) for j in range(m)]
        g=rng.normal(scale=.2,size=(c['pu'],c['pz']));mu=np.array([.1,-.1,.05,-.05]);dtrue=None
    else:
        dims={'heterogeneous':(1,4,2,1,4,2,3,1,2,4,1,3),
              'near_homogeneous':(2,)*12}[regime]
        strong=np.array([1.35,1.1,.85,.6,.45]);spec=[]
        for j in range(m):
            sc=np.sqrt(nlev[j]/np.median(nlev))*(1.+.015*(j-(m-1)/2))
            sj=np.zeros(klist[j]);sj[:dims[j]]=strong[:dims[j]]*sc;spec.append(sj)
        g=rng.normal(scale=.15,size=(c['pu'],c['pz']));mu=np.array([.1,-.05,.05,.05,-.05]);dtrue=dims
    t=[make_theta(nlev[j],c['pz'],spec[j],rng) for j in range(m)]
    return dict(theta=t,gamma=g,mu=mu,spec=spec,dtrue=dtrue)

def phi(z,weights):
    (w1,b1),(w2,b2)=weights
    return np.tanh(z @ w1.T + b1) @ w2.T + b2

def data(n,truth,weights,seed):
    c=CFG;rng=np.random.default_rng(seed)
    C=[rng.integers(0,c['nlev'][j],size=n) for j in range(c['m'])]
    U=rng.normal(size=(n,c['pu']))
    z=np.broadcast_to(truth['mu'],(n,c['pz'])).copy()+U@truth['gamma']
    for j in range(c['m']):z+=truth['theta'][j][C[j]]
    mean=phi(z,weights);Y=mean+c['noise_sd']*rng.normal(size=mean.shape)
    return C,U,Y

class Reference(nn.Module):
    def __init__(self):
        super().__init__();c=CFG
        self.emb=nn.ModuleList([nn.Embedding(n,c['pz']) for n in c['nlev']])
        for e in self.emb:nn.init.normal_(e.weight,std=.1 if c['sim']=='1' else .05)
        self.gamma=nn.Parameter(.08*torch.randn(c['pu'],c['pz']));self.mu=nn.Parameter(torch.zeros(c['pz']))
        self.l1=nn.Linear(c['pz'],c['pz']);self.l2=nn.Linear(c['pz'],c['q'])
        with torch.no_grad():
            for l in (self.l1,self.l2): l.weight.copy_(torch.linalg.qr(torch.randn(*l.weight.shape))[0]);l.bias.zero_()
    def forward(self,C,U):
        z=self.mu+U@self.gamma
        for j,e in enumerate(self.emb):z=z+e(C[j])
        return self.l2(torch.tanh(self.l1(z)))
    @torch.no_grad()
    def project(self):
        for e in self.emb:e.weight-=e.weight.mean(0,keepdim=True)
        for l in (self.l1,self.l2):
            a,_,v=torch.linalg.svd(l.weight,full_matrices=False);l.weight.copy_(a@v)

class Reduced(nn.Module):
    def __init__(self,d):
        super().__init__();c=CFG;self.d=d
        self.emb=nn.ModuleList([nn.Embedding(c['nlev'][j],d[j]) for j in range(c['m'])])
        for e in self.emb:nn.init.normal_(e.weight,std=.1)
        self.gamma=nn.Parameter(.08*torch.randn(c['pu'],c['pz']));self.mu=nn.Parameter(torch.zeros(c['pz']))
        self.l1=nn.Linear(sum(d)+c['pz'],c['pz']);self.l2=nn.Linear(c['pz'],c['q'])
    def forward(self,C,U):
        z=torch.cat([e(C[j]) for j,e in enumerate(self.emb)]+[self.mu+U@self.gamma],dim=1)
        return self.l2(torch.tanh(self.l1(z)))
    @torch.no_grad()
    def project(self):
        for e in self.emb:e.weight-=e.weight.mean(0,keepdim=True)

def tensors(t):
    C,U,Y=t
    return ([torch.as_tensor(z,dtype=torch.long,device=DEVICE) for z in C],
            torch.as_tensor(U,dtype=torch.float32,device=DEVICE),
            torch.as_tensor(Y,dtype=torch.float32,device=DEVICE))

def fit_model(train,kind,seed,d=None):
    c=CFG;C,U,Y=tensors(train)
    best=None;bestloss=np.inf
    restarts=c['ref_restarts'] if kind=='reference' else c['red_restarts']
    epochs=c['ref_epochs'] if kind=='reference' else c['red_epochs']
    for r in range(restarts):
        seed_all(seed+1000*r)
        net=(Reference() if kind=='reference' else Reduced(d)).to(DEVICE)
        opt=torch.optim.Adam(net.parameters(),lr=.005,weight_decay=5e-5 if kind=='reference' else 5e-4)
        for ep in range(epochs):
            opt.zero_grad();loss=F.mse_loss(net(C,U),Y);loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(),5.);opt.step();net.project()
            if ep%50==0:atomic_checkpoint()
        with torch.no_grad():result=float(F.mse_loss(net(C,U),Y).item())
        if result<bestloss:
            bestloss=result;best=net
        else:del net
    print(f'  {kind} d={d} training MSE={bestloss:.6f}',flush=True)
    return best,bestloss

@torch.no_grad()
def warm_reduced_from_reference(ref,d):
    """Construct exact/full or truncated SVD initialization in original Reduced class.

    Each centered reference embedding satisfies Theta_j = E_j V_j^T.
    The first-layer block W1 V_j therefore preserves the latent map.
    This changes only optimizer initialization, never posterior inference.
    """
    net=Reduced(d).to(DEVICE)
    net.gamma.copy_(torch.tensor(ref['gamma'],dtype=torch.float32,device=DEVICE))
    net.mu.copy_(torch.tensor(ref['mu'],dtype=torch.float32,device=DEVICE))
    (w1,b1),(w2,b2)=ref['omega']
    net.l1.bias.copy_(torch.tensor(b1,dtype=torch.float32,device=DEVICE))
    net.l2.weight.copy_(torch.tensor(w2,dtype=torch.float32,device=DEVICE))
    net.l2.bias.copy_(torch.tensor(b2,dtype=torch.float32,device=DEVICE))
    st=0
    for j,dj in enumerate(d):
        theta=ref['theta'][j]
        theta=theta-theta.mean(0,keepdims=True)
        u,ss,vt=np.linalg.svd(theta,full_matrices=False)
        E=u[:,:dj]*ss[:dj]
        net.emb[j].weight.copy_(torch.tensor(E,dtype=torch.float32,device=DEVICE))
        net.l1.weight[:,st:st+dj].copy_(torch.tensor(w1@vt[:dj].T,dtype=torch.float32,device=DEVICE))
        st+=dj
    net.l1.weight[:,st:st+CFG['pz']].copy_(torch.tensor(w1,dtype=torch.float32,device=DEVICE))
    return net

def fit_reduced_v7(ref,train,validation,d,seed):
    net=warm_reduced_from_reference(ref,d)
    Ct,Ut,Yt=tensors(train)
    with torch.no_grad():
        initial_train=float(F.mse_loss(net(Ct,Ut),Yt))
        initial_val=mse(validation[2],predict(net,validation))
    # At full capacity the reparameterization is (up to float32 roundoff)
    # the already optimized reference. Never spoil it with random refitting.
    if tuple(d)==tuple(CFG['klist']):
        expected=mse(validation[2],phi_from_ref_data(ref,validation))
        if abs(initial_val-expected)>2e-6:
            raise AssertionError('Full-dimension SVD does not reproduce the fitted reference')
        return net,dict(train=initial_train,val=initial_val,epochs=0,method='exact-reference-SVD')
    seed_all(seed)
    optimizer=torch.optim.Adam(net.parameters(),lr=REDUCED_LR,weight_decay=REDUCED_WEIGHT_DECAY)
    best_weights=copy.deepcopy(net.state_dict());best_val=initial_val;best_ep=0
    for ep in range(1,REDUCED_EPOCHS+1):
        optimizer.zero_grad()
        loss=F.mse_loss(net(Ct,Ut),Yt)
        loss.backward();torch.nn.utils.clip_grad_norm_(net.parameters(),5.)
        optimizer.step();net.project()
        if ep<=10 or ep%20==0:
            v=mse(validation[2],predict(net,validation))
            if v<best_val-1e-9:
                best_val=v;best_weights=copy.deepcopy(net.state_dict());best_ep=ep
        if ep-best_ep>=REDUCED_PATIENCE:break
    net.load_state_dict(best_weights);net.eval()
    with torch.no_grad(): final_train=float(F.mse_loss(net(Ct,Ut),Yt))
    return net,dict(train=final_train,val=best_val,epochs=best_ep,method='SVD-warm-start-validation')

def phi_from_ref_data(ref,dt):
    C,U,_=dt
    z=ref['mu']+U@ref['gamma']
    for j in range(CFG['m']):z+=ref['theta'][j][C[j]]
    return phi(z,ref['omega'])

def predict(net,dt):
    C,U,Y=tensors(dt)
    with torch.no_grad():return net(C,U).detach().cpu().numpy()

def reference_parameters(net):
    with torch.no_grad():
        th=[e.weight.detach().cpu().numpy().astype(float).copy() for e in net.emb]
        th=[v-v.mean(0,keepdims=True) for v in th]
        g=net.gamma.detach().cpu().numpy().astype(float).copy()
        mu=net.mu.detach().cpu().numpy().astype(float).copy()
        w=[(l.weight.detach().cpu().numpy().astype(float).copy(),l.bias.detach().cpu().numpy().astype(float).copy())
           for l in (net.l1,net.l2)]
    return dict(theta=th,gamma=g,mu=mu,omega=w)

def mse(y,yhat):return float(np.mean((y-yhat)**2))

# ---- Fixed-dimension posterior with integrated inverse-Gamma noise variance ----
S1A,S1B=2.,1.
A0,B0,POOL,AK,BK=2.,2.,4.,2.,1.
TAUG,TAUM=2.5,2.5
SIGMA_A,SIGMA_B=2.5,.02

def cliplogit(p):
    p=np.clip(p,1e-12,1-1e-12)
    return np.log(p)-np.log1p(-p)

def log_beta_logit(v,a,b):
    # Correct Jacobian: log Beta density on rho plus log |d rho / d v|.
    if a<=0 or b<=0:return -np.inf
    return float(a*log_expit(v)+b*log_expit(-v)-betaln(a,b))

def log_gamma_log(v,a,b):
    if abs(v)>50:return -np.inf
    return float(a*v-b*np.exp(v)+a*np.log(b)-gammaln(a))

def qr_haar(a):
    q,r=np.linalg.qr(a,mode='reduced')
    return q*np.where(np.diag(r)>=0.,1.,-1.)

def theta_from(s,j,refresh=None):
    # Computational cache ONLY: deterministic QR orientations depend solely on A,B.
    # The cache has no prior, likelihood, or statistical influence; always refresh
    # immediately when a proposed update changes the matching auxiliary matrix.
    c=CFG;K=c['klist'][j]
    if refresh in ('A','AB') or '_U' not in s:
        s['_U']=qr_haar(s['A'])[:,:K]
    if refresh in ('B','AB') or '_V' not in s:
        s['_V']=qr_haar(s['B'])[:,:K]
    u=s['_U'];v=s['_V']
    eig=np.empty(K);eig[0]=np.exp(s['x'])
    for h in range(1,K):eig[h]=eig[h-1]*expit(s['eta'][h-1])
    t=(c['bases'][j]@u)@np.diag(np.sqrt(eig))@v.T
    return t-t.mean(0,keepdims=True),eig

def start_spectrum(theta,j,rng,spread=1.0):
    c=CFG;K=c['klist'][j]
    u,sv,vt=np.linalg.svd(theta-theta.mean(0),full_matrices=False)
    s2=np.maximum(sv[:K]**2,1e-7)
    s2=np.minimum.accumulate(s2)
    A=c['bases'][j].T@u[:,:K];B=vt.T[:,:K]
    A=A+(0.05*spread)*rng.normal(size=A.shape);B=B+(0.05*spread)*rng.normal(size=B.shape)
    return dict(A=A,B=B,x=float(np.log(s2[0])+rng.normal(scale=.15*spread)),
                eta=cliplogit(np.clip(s2[1:]/s2[:-1],.001,.999))+rng.normal(scale=.3*spread,size=K-1))
def loglik(Y, yh):
    resid=Y-yh
    return float(-(SIGMA_A+.5*Y.size)*np.log(SIGMA_B+.5*np.sum(resid*resid)))

# JIT compiles ONLY deterministic likelihood arithmetic. Statistical updates,
# priors, exact slice kernels, replica-exchange probabilities and RNG stay Python.
# Neither fused routine changes the unnormalized integrated-variance target.
try:
    from numba import njit
    _NUMBA_AVAILABLE=True
except Exception:
    _NUMBA_AVAILABLE=False
    njit=None
_FAST_LL_MODE='numpy'
_FAST_LL_CHECKED=False
if _NUMBA_AVAILABLE:
    @njit(cache=False,fastmath=False)
    def _fused_ll_z(z,Y,w1,b1,w2,b2,alpha,beta):
        n,p=z.shape; hidden=np.empty(w1.shape[0]);ss=0.0
        for i in range(n):
            for h in range(w1.shape[0]):
                t=b1[h]
                for r in range(p): t+=z[i,r]*w1[h,r]
                hidden[h]=math.tanh(t)
            for q in range(w2.shape[0]):
                pred=b2[q]
                for h in range(w1.shape[0]): pred+=hidden[h]*w2[q,h]
                dif=Y[i,q]-pred;ss+=dif*dif
        return -(alpha+0.5*Y.size)*math.log(beta+0.5*ss)

    @njit(cache=False,fastmath=False)
    def _fused_ll_base(base,theta,cidx,Y,w1,b1,w2,b2,alpha,beta):
        n,p=base.shape; hidden=np.empty(w1.shape[0]);ss=0.0
        for i in range(n):
            row=cidx[i]
            for h in range(w1.shape[0]):
                t=b1[h]
                for r in range(p): t+=(base[i,r]+theta[row,r])*w1[h,r]
                hidden[h]=math.tanh(t)
            for q in range(w2.shape[0]):
                pred=b2[q]
                for h in range(w1.shape[0]): pred+=hidden[h]*w2[q,h]
                dif=Y[i,q]-pred;ss+=dif*dif
        return -(alpha+0.5*Y.size)*math.log(beta+0.5*ss)

def ll_from_z(z,Y,omega):
    if _FAST_LL_MODE=='numba':
        return float(_fused_ll_z(z,Y,*_as_weights(omega),SIGMA_A,SIGMA_B))
    return loglik(Y,phi(z,omega))

def ll_from_base(base,th,Cj,Y,omega,base_hidden=None):
    if _FAST_LL_MODE=='numba':
        return float(_fused_ll_base(base,th,Cj,Y,*_as_weights(omega),SIGMA_A,SIGMA_B))
    if base_hidden is not None:
        (w1,b1),(w2,b2)=omega
        predicted=np.tanh(base_hidden+(th@w1.T)[Cj])@w2.T+b2
        return loglik(Y,predicted)
    return loglik(Y,phi(base+th[Cj],omega))

def _as_weights(omega):
    (w1,b1),(w2,b2)=omega
    return w1,b1,w2,b2

def configure_fast_likelihood(train,omega):
    # Validate BOTH fused forms against the unmodified NumPy likelihood,
    # at the actual training size, then use JIT only when locally faster.
    global _FAST_LL_MODE,_FAST_LL_CHECKED
    if _FAST_LL_CHECKED: return
    _FAST_LL_CHECKED=True
    if not _NUMBA_AVAILABLE:
        print('FAST LL: Numba not installed; using preprojected cached NumPy likelihood.',flush=True)
        return
    import time as _clock
    C,U,Y=train
    rng=np.random.default_rng(47017)
    z=rng.normal(scale=.5,size=(Y.shape[0],CFG['pz']))
    th=rng.normal(scale=.05,size=(CFG['nlev'][0],CFG['pz']))
    # Warm JIT before timing; no random seed/state of the sampler is touched.
    w=_as_weights(omega)
    a=_fused_ll_z(z,Y,*w,SIGMA_A,SIGMA_B)
    b=_fused_ll_base(z,th,C[0],Y,*w,SIGMA_A,SIGMA_B)
    a0=loglik(Y,phi(z,omega))
    b0=loglik(Y,phi(z+th[C[0]],omega))
    if not np.isclose(a,a0,rtol=0,atol=5e-7) or not np.isclose(b,b0,rtol=0,atol=5e-7):
        print('FAST LL: validation failed, retaining NumPy evaluator.',a-a0,b-b0,flush=True)
        return
    N=20
    t=_clock.perf_counter()
    for _ in range(N):
        loglik(Y,phi(z+th[C[0]],omega))
    slow=(_clock.perf_counter()-t)/N
    bh=z@omega[0][0].T+omega[0][1]
    t=_clock.perf_counter()
    for _ in range(N):
        ll_from_base(z,th,C[0],Y,omega,bh)
    numpy_cached=(_clock.perf_counter()-t)/N
    t=_clock.perf_counter()
    for _ in range(N):
        _fused_ll_base(z,th,C[0],Y,*w,SIGMA_A,SIGMA_B)
    fast=(_clock.perf_counter()-t)/N
    # Prefer the fastest VERIFIED evaluator at this machine's actual n=1200.
    if fast<0.98*numpy_cached:
        _FAST_LL_MODE='numba'
    print(f'FAST LL: backend={_FAST_LL_MODE} | original {slow*1e6:.1f}us, '
          f'cached NumPy {numpy_cached*1e6:.1f}us, compiled {fast*1e6:.1f}us per candidate.',flush=True)

def draw_sigma2(Y,yh,rng):
    a=SIGMA_A+.5*Y.size;b=SIGMA_B+.5*np.sum((Y-yh)**2)
    return float(1./rng.gamma(a,1./b))

def ess_block(current,log_L,rng,prior_sd=1.,max_evals=300):
    # Elliptical slice; valid for zero-mean isotropic Gaussian prior coordinates.
    nu=prior_sd*rng.normal(size=current.shape)
    level=log_L(current)+np.log(rng.random())
    angle=rng.uniform(0.,2*np.pi);low=angle-2*np.pi;high=angle
    for evaluations in range(1,max_evals+1):
        proposed=current*np.cos(angle)+nu*np.sin(angle)
        if log_L(proposed)>=level:return proposed,evaluations
        if angle<0:low=angle
        else:high=angle
        angle=rng.uniform(low,high)
    raise RuntimeError('Elliptical slice exceeded bracket budget; stop and inspect sampler.')

def slice1(x,ld,rng,w=.5,max_evals=400):
    current=ld(x)
    if not np.isfinite(current):raise FloatingPointError('Nonfinite current log density in slice sampler.')
    level=current+np.log(rng.random());offset=rng.random()
    L=x-w*offset;R=L+w
    J=int(rng.integers(20));K=19-J; ne=0
    while J>0 and ld(L)>level:L-=w;J-=1;ne+=1
    while K>0 and ld(R)>level:R+=w;K-=1;ne+=1
    for t in range(max_evals):
        xnew=rng.uniform(L,R);v=ld(xnew);ne+=1
        if v>=level:return float(xnew),ne
        if xnew<x:L=xnew
        else:R=xnew
    raise RuntimeError('Scalar slice exceeded shrinkage budget.')

def current_hyper(h):
    return expit(h['z0']),expit(h['zj']),np.exp(h['lk'])

def hierarchical_logprior(h,states):
    means0,means,kappa=current_hyper(h)
    if not np.isfinite(kappa) or kappa>1e5:return -np.inf
    lp=log_beta_logit(h['z0'],A0,B0)+log_gamma_log(h['lk'],AK,BK)
    for j in range(CFG['m']):
        lp+=log_beta_logit(h['zj'][j],POOL*means0,POOL*(1-means0))
        a=kappa*means[j];b=kappa*(1-means[j])
        for eta in states[j]['eta']:
            lp+=log_beta_logit(eta,a,b)
    return float(lp)

# Faster exact conditional log densities: terms constant in the coordinate
# under update are omitted, as permitted by slice sampling. No approximation.
def local_zj_logprior(v,j,h,states):
    h0=expit(h['z0']);rho=expit(v);kappa=np.exp(h['lk'])
    if not np.isfinite(kappa) or kappa>1e5:return -np.inf
    a=kappa*rho;b=kappa*(1.-rho)
    et=states[j]['eta']
    return float(log_beta_logit(v,POOL*h0,POOL*(1.-h0))+
                 np.sum(a*log_expit(et)+b*log_expit(-et))-len(et)*betaln(a,b))

def local_z0_logprior(v,h):
    rho0=expit(v)
    zj=h['zj']
    aa=POOL*rho0;bb=POOL*(1.-rho0)
    return float(log_beta_logit(v,A0,B0)+
                 np.sum(aa*log_expit(zj)+bb*log_expit(-zj))-len(zj)*betaln(aa,bb))

def local_kappa_logprior(v,h,states):
    if abs(v)>50:return -np.inf
    kappa=np.exp(v)
    if not np.isfinite(kappa) or kappa>1e5:return -np.inf
    means=expit(h['zj'])
    lp=log_gamma_log(v,AK,BK)
    for j in range(CFG['m']):
        et=states[j]['eta'];a=kappa*means[j];b=kappa*(1.-means[j])
        lp+=float(np.sum(a*log_expit(et)+b*log_expit(-et))-len(et)*betaln(a,b))
    return float(lp)

def start_chain(ref,train,seed,spread=1.0):
    C,U,Y=train;rng=np.random.default_rng(seed)
    s=[start_spectrum(ref['theta'][j],j,rng,spread=spread) for j in range(CFG['m'])]
    th=[theta_from(s[j],j)[0] for j in range(CFG['m'])]
    g=ref['gamma'].copy()+(0.025*spread)*rng.normal(size=ref['gamma'].shape)
    mu=ref['mu'].copy()+(0.025*spread)*rng.normal(size=ref['mu'].shape)
    z=mu+U@g
    for j in range(CFG['m']):z+=th[j][C[j]]
    yh=phi(z,ref['omega']);ll=loglik(Y,yh)
    h=dict(z0=float(cliplogit(rng.beta(2,2))),
           zj=cliplogit(np.clip(rng.beta(4,4,size=CFG['m']),.001,.999)),
           lk=float(np.log(rng.gamma(2,1))))
    return dict(it=0,rng=rng,s=s,th=th,g=g,mu=mu,z=z,yh=yh,ll=ll,h=h,
                traces={'s2':[],'bar':[],'bar0':[],'kappa':[],'sigma2':[],'ll':[],
                        'gamma':[],'mu':[]},
                effort=np.zeros((CFG['m'],3),dtype=np.int64),
                updates=np.zeros((CFG['m'],3),dtype=np.int64),
                h_effort=np.zeros(3,dtype=np.int64),h_updates=np.zeros(3,dtype=np.int64),
                joint_effort=0,joint_updates=0,global_effort=0,global_updates=0,
                refresh_try=np.zeros(CFG['m'],dtype=np.int64),refresh_accept=np.zeros(CFG['m'],dtype=np.int64),
                 tail_mh_try=np.zeros((CFG['m'],max(CFG['klist'])-1),dtype=np.int64),
                 tail_mh_accept=np.zeros((CFG['m'],max(CFG['klist'])-1),dtype=np.int64),
                 tail_mh_type_try=np.zeros(2,dtype=np.int64),
                 tail_mh_type_accept=np.zeros(2,dtype=np.int64),
                spectral_block_evals=np.zeros(CFG['m'],dtype=np.int64),
                spectral_block_updates=np.zeros(CFG['m'],dtype=np.int64),
                coupled_tail_evals=np.zeros(CFG['m'],dtype=np.int64),
                coupled_tail_updates=np.zeros(CFG['m'],dtype=np.int64))
def sampler_step(p,train,omega,beta=1.0,record=True,global_move=False,
                  prior_refresh=False,spectral_block=False,coupled_tail=False,predictor_subset=None,
                 targeted_tail_mh=False):
    # Every component below is an exact conditional/ESS/MH update for the
    # same beta-tempered posterior. A hot predictor_subset merely composes
    # fewer invariant kernels; no pseudo-likelihood or approximate gradients.
    C,U,Y=train;rng=p['rng']
    active=range(CFG['m']) if predictor_subset is None else predictor_subset
    for j in active:
        sj=p['s'][j];Cj=C[j]
        base_z=p['z']-p['th'][j][Cj]
        (w1,b1),_=omega
        base_hidden=base_z@w1.T+b1 if _FAST_LL_MODE=='numpy' else None
        # While updating this predictor the contribution from every OTHER
        # predictor is fixed; update latent state only after finishing this j.
        for bi,key in enumerate(('A','B')):
            old=sj[key].copy();accepted={}
            def evaluate(a):
                sj[key]=a
                th,_=theta_from(sj,j,refresh=key)
                ll=ll_from_base(base_z,th,Cj,Y,omega,base_hidden)
                accepted.update(th=th,ll=ll)
                return beta*ll
            new,ne=ess_block(old,evaluate,rng)
            sj[key]=new
            p['ll']=accepted['ll'];p['th'][j]=accepted['th']
            p['effort'][j,bi]+=ne;p['updates'][j,bi]+=1
        _,means,kappa=current_hyper(p['h'])
        for idx in range(CFG['klist'][j]):
            is_x=(idx==0);ii=idx-1
            old=float(sj['x'] if is_x else sj['eta'][ii])
            accepted={}
            def score(value):
                if not np.isfinite(value) or (is_x and abs(value)>60) or ((not is_x) and abs(value)>700):return -np.inf
                if is_x:sj['x']=value
                else:sj['eta'][ii]=value
                th,_=theta_from(sj,j)
                ll=ll_from_base(base_z,th,Cj,Y,omega,base_hidden)
                accepted.update(th=th,ll=ll)
                prior=log_gamma_log(value,S1A,S1B) if is_x else log_beta_logit(value,kappa*means[j],kappa*(1-means[j]))
                return prior+beta*ll
            new,ne=slice1(old,score,rng,w=.25 if is_x else (TAIL_RATIO_SLICE_WIDTH if (j in TAIL_FOCUS and ii in TAIL_FOCUS[j]) else .7))
            if is_x:sj['x']=new
            else:sj['eta'][ii]=new
            p['th'][j]=accepted['th'];p['ll']=accepted['ll']
            p['effort'][j,2]+=ne;p['updates'][j,2]+=1
        # Reconstruct z ONCE per predictor, rather than after every ESS/slice.
        p['z']=base_z+p['th'][j][Cj]
    if prior_refresh:
        _,means,kappa=current_hyper(p['h'])
        for j in active:
            sj=p['s'][j];old_eta=sj['eta'].copy()
            if old_eta.size==0:continue
            aa=max(1e-7,kappa*means[j]);bb=max(1e-7,kappa*(1-means[j]))
            rho_prop=np.clip(rng.beta(aa,bb,size=old_eta.size),1e-12,1-1e-12)
            sj['eta']=cliplogit(rho_prop)
            th,_=theta_from(sj,j)
            base_z=p['z']-p['th'][j][C[j]]
            bh=base_z@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
            new_ll=ll_from_base(base_z,th,C[j],Y,omega,bh)
            p['refresh_try'][j]+=1
            if np.log(rng.random())<min(0.,beta*(new_ll-p['ll'])):
                p['refresh_accept'][j]+=1
                p['z']=base_z+th[C[j]];p['th'][j]=th;p['ll']=new_ll
            else:sj['eta']=old_eta
    # Robust exact joint Beta-ratio update. q is a fixed mixture of Beta(a,b+lambda).
    # The log of q/prior is computed analytically, and corrected in MH.
    if spectral_block:
        for j in active:
            if len(p['s'][j]['eta']):
                joint_tilted_beta_mh(p,train,omega,beta,j)

    # Coupled exact update of one weak ratio and its two Gaussian orientation columns.
    # The Gaussian proposal is the original auxiliary prior, and its density cancels.
    if coupled_tail:
        for j in active:
            if CFG['klist'][j]>=2:
                coupled_tilted_beta_orientation_mh(p,train,omega,beta,j)

    oldg=p['g'].copy();oldmu=p['mu'].copy()
    ng=oldg.size;joint_old=np.concatenate([oldg.reshape(-1)/TAUG,oldmu/TAUM])
    accepted={}
    def joint_ll(a):
        candg=(a[:ng]*TAUG).reshape(oldg.shape);candmu=a[ng:]*TAUM
        zz=p['z']+U@(candg-oldg)+(candmu-oldmu)
        ll=ll_from_z(zz,Y,omega)
        accepted.update(z=zz,ll=ll)
        return beta*ll
    joint_new,joint_ne=ess_block(joint_old,joint_ll,rng,prior_sd=1.)
    p['g']=(joint_new[:ng]*TAUG).reshape(oldg.shape)
    p['mu']=joint_new[ng:]*TAUM
    p['z']=accepted['z'];p['ll']=accepted['ll']
    p['joint_effort']+=joint_ne;p['joint_updates']+=1
    if global_move:
        j=int(rng.integers(CFG['m']));sj=p['s'][j]
        oldA=sj['A'].copy();oldB=sj['B'].copy()
        oldg=p['g'].copy();oldmu=p['mu'].copy()
        na=oldA.size;nb=oldB.size;ng=oldg.size
        block_old=np.concatenate([oldA.reshape(-1),oldB.reshape(-1),oldg.reshape(-1)/TAUG,oldmu/TAUM])
        accepted={}
        base_z=p['z']-p['th'][j][C[j]]
        def block_ll(a):
            aa=a[:na].reshape(oldA.shape);bb=a[na:na+nb].reshape(oldB.shape)
            newg=(a[na+nb:na+nb+ng]*TAUG).reshape(oldg.shape)
            newmu=a[na+nb+ng:]*TAUM
            sj['A']=aa;sj['B']=bb
            th,_=theta_from(sj,j,refresh='AB')
            zz=base_z+th[C[j]]+U@(newg-oldg)+(newmu-oldmu)
            ll=ll_from_z(zz,Y,omega)
            accepted.update(th=th,z=zz,ll=ll)
            return beta*ll
        new,ne=ess_block(block_old,block_ll,rng,prior_sd=1.,max_evals=500)
        sj['A']=new[:na].reshape(oldA.shape);sj['B']=new[na:na+nb].reshape(oldB.shape)
        p['g']=(new[na+nb:na+nb+ng]*TAUG).reshape(oldg.shape)
        p['mu']=new[na+nb+ng:]*TAUM
        p['z']=accepted['z'];p['th'][j]=accepted['th'];p['ll']=accepted['ll']
        p['global_effort']+=ne;p['global_updates']+=1
    h=p['h'];states=p['s']
    # Slice samplers only need a conditional log density up to a CONSTANT.
    # This avoids rescoring all 12 predictor priors for every hyper mean move.
    for typ in ('z0','zj','lk'):
        if typ=='zj':
            for j in range(CFG['m']):
                old=float(h['zj'][j])
                def ld(v):return local_zj_logprior(v,j,h,states)
                new,ne=slice1(old,ld,rng,w=.5)
                h['zj'][j]=new;p['h_effort'][1]+=ne;p['h_updates'][1]+=1
        else:
            old=float(h[typ])
            if typ=='z0':
                def ld(v):return local_z0_logprior(v,h)
            else:
                def ld(v):return local_kappa_logprior(v,h,states)
            new,ne=slice1(old,ld,rng,w=.3 if typ=='lk' else .5)
            h[typ]=new;i=0 if typ=='z0' else 2
            p['h_effort'][i]+=ne;p['h_updates'][i]+=1
    # Full-support, prior-corrected independent tail refresh BEFORE recording traces.
    if targeted_tail_mh:
        tail_mh_sweep(p,train,omega,beta,p['it']+1)
    # Cache exact predictions once per sweep, essential for sigma2 draw/checkpoint.
    p['yh']=phi(p['z'],omega)
    p['ll']=loglik(Y,p['yh'])
    p['it']+=1
    it=p['it']
    if record and it>MCMC_BURN and (it-MCMC_BURN)%MCMC_THIN==0:
        sp=np.array([np.pad(theta_from(p['s'][j],j)[1],(0,max(CFG['klist'])-CFG['klist'][j]),constant_values=np.nan)
                     for j in range(CFG['m'])])
        h0,means,kappa=current_hyper(p['h']);t=p['traces']
        t['s2'].append(sp)
        t['bar'].append(means.copy());t['bar0'].append(float(h0));t['kappa'].append(float(kappa))
        t['sigma2'].append(draw_sigma2(Y,p['yh'],rng));t['ll'].append(p['ll'])
        t['gamma'].append(p['g'].copy());t['mu'].append(p['mu'].copy())

# ---- Exact prior-weighted mixture proposals for near-zero Beta ratios ----
# q(rho|a,b) = sum_h w_h Beta(rho|a,b+lambda_h).
# q/prior = H(rho) = sum_h w_h B(a,b)/B(a,b+lambda_h) (1-rho)^lambda_h.
# Thus all target Beta normalizing constants cancel in MH, with log acceptance
# beta*(new_ll-old_ll) + log H(old_rho) - log H(new_rho).
# The lambda grid and weights are frozen before sampling, with lambda=0
# guaranteeing full support. The hierarchical Beta prior is unchanged.
_LAM=np.asarray(TAIL_TILT_LAMBDAS,dtype=float)
_W=np.asarray(TAIL_TILT_WEIGHTS,dtype=float)
if not np.isclose(_W.sum(),1.,rtol=0,atol=1e-12):
    raise ValueError('Tilt mixture weights must sum exactly to 1')

def log_tilt_factor(rho,a,b):
    if not (0.<rho<1.) or a<=0 or b<=0:return -np.inf
    return float(logsumexp(np.log(_W)+betaln(a,b)-betaln(a,b+_LAM)
                           +_LAM*np.log1p(-rho)))

def tilted_proposal(rng,a,b):
    ind=int(rng.choice(len(_W),p=_W))
    rho=float(rng.beta(a,b+float(_LAM[ind])))
    # No numerical Beta-CDF inverse or forced 1e-13 CDF floor.
    rho=float(np.clip(rho,1e-300,1.-1e-15))
    return rho,float(math.log(rho)-math.log1p(-rho))

def tail_beta_mh_coordinate(p,train,omega,beta,j,k):
    C,U,Y=train;sj=p['s'][j];rng=p['rng']
    if k<0 or k>=len(sj['eta']):return False
    _,means,kappa=current_hyper(p['h'])
    a=float(kappa*means[j]);b=float(kappa*(1.-means[j]))
    old_eta=float(sj['eta'][k]);old_rho=float(expit(old_eta))
    new_rho,new_eta=tilted_proposal(rng,a,b)
    log_ratio=log_tilt_factor(old_rho,a,b)-log_tilt_factor(new_rho,a,b)
    base=p['z']-p['th'][j][C[j]]
    sj['eta'][k]=new_eta
    theta,_=theta_from(sj,j)
    hidden=base@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
    ll=ll_from_base(base,theta,C[j],Y,omega,hidden)
    p['tail_mh_try'][j,k]+=1;p['tail_mh_type_try'][0]+=1
    if np.log(rng.random())<min(0.,beta*(ll-p['ll'])+log_ratio):
        p['tail_mh_accept'][j,k]+=1;p['tail_mh_type_accept'][0]+=1
        p['th'][j]=theta;p['z']=base+theta[C[j]];p['ll']=ll
        return True
    sj['eta'][k]=old_eta
    return False

def joint_tilted_beta_mh(p,train,omega,beta,j):
    C,U,Y=train;sj=p['s'][j];rng=p['rng']
    _,means,kappa=current_hyper(p['h']);a=float(kappa*means[j]);b=float(kappa*(1.-means[j]))
    old=sj['eta'].copy();new=[];corr=0.
    for old_eta in old:
        rr,ee=tilted_proposal(rng,a,b)
        corr+=log_tilt_factor(float(expit(old_eta)),a,b)-log_tilt_factor(rr,a,b)
        new.append(ee)
    base=p['z']-p['th'][j][C[j]]
    sj['eta']=np.asarray(new)
    theta,_=theta_from(sj,j)
    hidden=base@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
    ll=ll_from_base(base,theta,C[j],Y,omega,hidden)
    p['spectral_block_evals'][j]+=1;p['spectral_block_updates'][j]+=1
    if np.log(rng.random())<min(0.,beta*(ll-p['ll'])+corr):
        p['th'][j]=theta;p['z']=base+theta[C[j]];p['ll']=ll
        return True
    sj['eta']=old
    return False

def coupled_tilted_beta_orientation_mh(p,train,omega,beta,j):
    C,U,Y=train;sj=p['s'][j];rng=p['rng'];K=CFG['klist'][j]
    k=int(1 if (j==7 or j==11) else (K-1 if rng.random()<.65 else rng.integers(1,K)))
    _,means,kappa=current_hyper(p['h']);a=float(kappa*means[j]);b=float(kappa*(1.-means[j]))
    old_eta=float(sj['eta'][k-1]);old_A=sj['A'][:,k].copy();old_B=sj['B'][:,k].copy()
    rho,eta=tilted_proposal(rng,a,b)
    corr=log_tilt_factor(float(expit(old_eta)),a,b)-log_tilt_factor(rho,a,b)
    base=p['z']-p['th'][j][C[j]]
    sj['A'][:,k]=rng.normal(size=old_A.shape)
    sj['B'][:,k]=rng.normal(size=old_B.shape)
    sj['eta'][k-1]=eta
    theta,_=theta_from(sj,j,refresh='AB')
    hidden=base@omega[0][0].T+omega[0][1] if _FAST_LL_MODE=='numpy' else None
    ll=ll_from_base(base,theta,C[j],Y,omega,hidden)
    p['coupled_tail_evals'][j]+=1;p['coupled_tail_updates'][j]+=1
    if np.log(rng.random())<min(0.,beta*(ll-p['ll'])+corr):
        p['th'][j]=theta;p['z']=base+theta[C[j]];p['ll']=ll
        return True
    sj['A'][:,k]=old_A;sj['B'][:,k]=old_B;sj['eta'][k-1]=old_eta
    theta_from(sj,j,refresh='AB')  # restore QR cache as well as auxiliary state
    return False

def tail_mh_sweep(p,train,omega,beta,i):
    # Fixed update schedule: preserves the beta-tempered posterior at every step.
    for repeat in range(TAIL_PRIORITY_REPEATS):
        for j,k in TAIL_PRIORITY:
            tail_beta_mh_coordinate(p,train,omega,beta,j,k)
    if i%TAIL_PRIOR_MH_EVERY==0:
        for j,coords in TAIL_FOCUS.items():
            k=coords[(i//TAIL_PRIOR_MH_EVERY-1)%len(coords)]
            if (j,k) not in TAIL_PRIORITY:
                tail_beta_mh_coordinate(p,train,omega,beta,j,k)

# ---- Independent parallel-tempering ladders; exact cold target at beta=1 ----
PT_STATE_FIELDS=('s','th','g','mu','z','yh','ll','h')
def make_ladder(ref,train,seed):
    betas=np.geomspace(1.,PT_MIN_BETA,PT_TEMPS)
    replicas=[start_chain(ref,train,seed+1009*k,spread=(1.+1.5*k/max(1,PT_TEMPS-1)))
              for k in range(PT_TEMPS)]
    return dict(it=0,betas=betas,log_gaps=-np.diff(np.log(betas)),replicas=replicas,
                tries=np.zeros(PT_TEMPS-1,dtype=np.int64),
                accepts=np.zeros(PT_TEMPS-1,dtype=np.int64),
                window_tries=np.zeros(PT_TEMPS-1,dtype=np.int64),
                window_accepts=np.zeros(PT_TEMPS-1,dtype=np.int64),
                labels=np.arange(PT_TEMPS,dtype=np.int64),
                seen_hot=np.zeros(PT_TEMPS,dtype=bool),roundtrips=0,
                cold_ll_initial=float(replicas[0]['ll']))

def pt_exchange(ladder,rng):
    # Alternating even/odd adjacent swaps. Uses whole posterior state, NOT RNG/traces.
    parity=(ladder['it']//PT_SWAP_EVERY)%2
    for k in range(parity,PT_TEMPS-1,2):
        r1=ladder['replicas'][k];r2=ladder['replicas'][k+1]
        b1=float(ladder['betas'][k]);b2=float(ladder['betas'][k+1])
        log_accept=(b1-b2)*(float(r2['ll'])-float(r1['ll']))
        ladder['tries'][k]+=1;ladder['window_tries'][k]+=1
        if np.log(rng.random())<min(0.,log_accept):
            for key in PT_STATE_FIELDS:
                r1[key],r2[key]=r2[key],r1[key]
            ladder['labels'][k],ladder['labels'][k+1]=ladder['labels'][k+1],ladder['labels'][k]
            ladder['accepts'][k]+=1;ladder['window_accepts'][k]+=1
    ladder['seen_hot'][ladder['labels'][-1]]=True
    cold_label=int(ladder['labels'][0])
    if ladder['seen_hot'][cold_label]:
        ladder['roundtrips']+=1
        ladder['seen_hot'][cold_label]=False

def pt_adapt(ladder):
    # Adapt only during burn in; cumulative beta endpoints fixed.
    trials=ladder['window_tries'];acc=ladder['window_accepts']
    frac=np.divide(acc,np.maximum(1,trials))
    eligible=trials>0
    logg=np.log(ladder['log_gaps'])
    if eligible.any():
        rate=0.5/np.sqrt(1.+ladder['it']/float(PT_ADAPT_EVERY))
        logg[eligible]+=rate*(frac[eligible]-PT_TARGET_SWAP)
        vals=np.exp(logg-np.max(logg));gaps=vals/vals.sum()*(-np.log(PT_MIN_BETA))
        ladder['log_gaps']=gaps
        ladder['betas']=np.r_[1.,np.exp(-np.cumsum(gaps))]
    ladder['window_tries'][:]=0;ladder['window_accepts'][:]=0

def pt_one_sweep(ladder,train,omega):
    i=ladder['it']+1
    for k,rep in enumerate(ladder['replicas']):
        if k>0 and i%PT_HOT_SWEEP_EVERY:continue
        # Rotate disjoint 4-predictor blocks across successive hot sweeps;
        # each block update leaves the *full* tempered posterior invariant.
        if k==0:
            subset=None
        else:
            m=CFG['m']
            st=((i//PT_HOT_SWEEP_EVERY-1)*PT_HOT_PREDICTORS+(k%3)*PT_HOT_PREDICTORS)%m
            subset=tuple((st+r)%m for r in range(PT_HOT_PREDICTORS))
        sampler_step(rep,train,omega,beta=float(ladder['betas'][k]),record=(k==0),
                     global_move=(i%PT_GLOBAL_MOVE_EVERY==0),
                     prior_refresh=(i%PT_PRIOR_REFRESH_EVERY==0),
                     spectral_block=(k==0 and i%PT_SPECTRAL_BLOCK_EVERY==0),
                     coupled_tail=(k==0 and i%PT_COUPLED_TAIL_EVERY==0),
                     predictor_subset=subset,
                     targeted_tail_mh=(k==0 or (k>0 and i%4==0)))
    ladder['it']=i
    # A lightweight cache-consistency audit detects invalid state exchanges.
    if i in (1, 10) or i%1000==0:
        C,U,Y=train
        for rep in ladder['replicas']:
            check_z=rep['mu']+U@rep['g']
            for j in range(CFG['m']): check_z+=rep['th'][j][C[j]]
            fresh_ll=loglik(Y,phi(check_z,omega))
            if not np.allclose(rep['ll'],fresh_ll,rtol=0,atol=1e-7):
                raise AssertionError('Parallel tempering cached log-likelihood mismatch')
    if i%PT_SWAP_EVERY==0:
        pt_exchange(ladder,ladder['replicas'][0]['rng'])
    if i==1000:
        rates=ladder['accepts']/np.maximum(1,ladder['tries'])
        if np.min(rates)<PT_MIN_SWAP_WARN:
            print('EARLY PT ALERT: weakest adjacent swap rate',round(float(np.min(rates)),3),
                  'after 1000 iterations. Dense ladder may still need retuning.',flush=True)
    if i<=MCMC_BURN and i%PT_ADAPT_EVERY==0:
        pt_adapt(ladder)

def pt_report(ladders):
    print('PARALLEL TEMPERING (replica exchanges; cold posterior remains beta=1):',flush=True)
    rows=[]
    for ci,L in enumerate(ladders):
        for k in range(PT_TEMPS-1):
            tries=int(L['tries'][k]);ok=int(L['accepts'][k])
            rows.append(dict(chain=ci+1,pair=f'{k+1}-{k+2}',
                             beta1=float(L['betas'][k]),beta2=float(L['betas'][k+1]),
                             accepted=ok,attempts=tries,rate=(ok/tries if tries else np.nan)))
    table=pd.DataFrame(rows)
    print(table.round(4).to_string(index=False),flush=True)
    print('Roundtrips (hot->cold):',[int(L['roundtrips']) for L in ladders],flush=True)
    rate=table['rate'].to_numpy()
    bottleneck=~np.isfinite(rate)|(rate<PT_MIN_SWAP_WARN)
    if bottleneck.any():
        print('WARNING: parallel-tempering swap bottleneck (below 8% or undefined).',flush=True)
    print('Cold-chain JOINT (Gamma,mu) ESS evals/update:',
          [round(L['replicas'][0]['joint_effort']/max(1,L['replicas'][0]['joint_updates']),2) for L in ladders],flush=True)
    print('Per-predictor conditional-prior block refresh acceptance (cold only):',
          np.round(sum((L['replicas'][0]['refresh_accept'] for L in ladders)) /
                   np.maximum(1,sum((L['replicas'][0]['refresh_try'] for L in ladders))),3).tolist(),flush=True)
    print('Cold-chain GLOBAL Gaussian-block ESS evals/update:',
          [round(L['replicas'][0]['global_effort']/max(1,L['replicas'][0]['global_updates']),2) for L in ladders],flush=True)
    be=sum((L['replicas'][0]['spectral_block_evals'] for L in ladders));bn=sum((L['replicas'][0]['spectral_block_updates'] for L in ladders))
    ce=sum((L['replicas'][0]['coupled_tail_evals'] for L in ladders));cn=sum((L['replicas'][0]['coupled_tail_updates'] for L in ladders))
    print('Cold joint tilted-Beta proposals/update:',np.round(be/np.maximum(bn,1),2).tolist(),flush=True)
    print('Cold coupled-tail MH proposals/update:',np.round(ce/np.maximum(cn,1),2).tolist(),flush=True)
    tr=sum((L['replicas'][0]['tail_mh_try'] for L in ladders))
    ac=sum((L['replicas'][0]['tail_mh_accept'] for L in ladders))
    print('TARGETED TAIL tilted-Beta mixture independence MH acceptance (j,ratio_index,accepted/tried):',flush=True)
    for j in sorted(TAIL_FOCUS):
        for k in TAIL_FOCUS[j]:
            if tr[j,k]:print(f' j={j+1}, ratio={k+1}: {ac[j,k]}/{tr[j,k]} = {ac[j,k]/tr[j,k]:.3f}',flush=True)
    by_type_tr=sum((L['replicas'][0]['tail_mh_type_try'] for L in ladders))
    by_type_ac=sum((L['replicas'][0]['tail_mh_type_accept'] for L in ladders))
    print('MH proposal acceptance [tilted-Beta mixture]:',
          np.round(by_type_ac/np.maximum(by_type_tr,1),3).tolist(),flush=True)
    # Bottlenecks inform troubleshooting, but only target-posterior Rhat/ESS are mandatory convergence gates.
    return table

# ---- Honest convergence checks; invariants (spectral tails) are primary diagnostic targets ----
def diag_one(arr):
    ar=np.asarray(arr,dtype=float)
    if ar.ndim!=2 or ar.shape[0]<2 or ar.shape[1]<8:return (np.nan,np.nan,np.nan,np.nan)
    if not np.isfinite(ar).all():return (np.nan,np.nan,np.nan,np.nan)
    if any(np.ptp(ar[i])<1e-12 for i in range(ar.shape[0])):return (np.nan,np.nan,np.nan,np.nan)
    try:
        return (float(az.rhat(ar,method='rank')),
                float(az.ess(ar,method='bulk')),
                float(az.ess(ar,method='tail',prob=(.05,.95))),
                float(np.asarray(az.mcse(ar,method='mean')).item()))
    except Exception:
        return (np.nan,np.nan,np.nan,np.nan)

def diagnostics(chains, label, target_iters):
    c=CFG;K=c['klist'];m=c['m']
    s=np.stack([np.asarray(p['traces']['s2']) for p in chains])
    h=np.stack([np.asarray(p['traces']['bar']) for p in chains]);h0=np.stack([p['traces']['bar0'] for p in chains]);kap=np.stack([p['traces']['kappa'] for p in chains])
    ll=np.stack([p['traces']['ll'] for p in chains]);sig=np.stack([p['traces']['sigma2'] for p in chains])
    g=np.stack([np.asarray(p['traces']['gamma']) for p in chains]);mu=np.stack([np.asarray(p['traces']['mu']) for p in chains])
    rows=[]
    def add(name,v,core=True):
        rh,eb,et,mc=diag_one(v)
        rows.append(dict(quantity=name,rhat=rh,ess_bulk=eb,ess_tail=et,mcse=mc,mean=float(np.mean(v)),sd=float(np.std(v)),core=core))
    for j in range(m):
        add(f'j{j+1}:bar_rho',h[:,:,j])
        for k in range(K[j]):add(f'j{j+1}:s2_{k+1}',s[:,:,j,k])
        for d in range(1,K[j]):add(f'j{j+1}:tail_{d}',np.sum(s[:,:,j,d:K[j]],axis=-1))
        # This is NOT a posterior rank probability; derived effective-rank diagnostic only.
    add('hyper:bar0',h0);add('hyper:kappa',kap);add('sigma2',sig);add('log_collapsed_like',ll)
    for ii in range(c['pu']):
        for kk in range(c['pz']):add(f'gamma_{ii+1}_{kk+1}',g[:,:,ii,kk])
    for kk in range(c['pz']):add(f'mu_{kk+1}',mu[:,:,kk])
    df=pd.DataFrame(rows)
    okmask=df['core']
    fail=okmask & ((~np.isfinite(df.rhat)) | (df.rhat>RHAT_LIMIT) |
                   (~np.isfinite(df.ess_bulk)) | (df.ess_bulk<ESS_BULK_MIN) |
                   (~np.isfinite(df.ess_tail)) | (df.ess_tail<ESS_TAIL_MIN))
    print('\n'+'='*80,flush=True)
    print('MCMC DIAGNOSTICS:',label,'| per chain iters=',target_iters,'| retained=',s.shape[1], ' | chains=',len(chains),flush=True)
    print('Rhat and ESS are split-rank ArviZ estimates. Undefined values are not treated as successes.')
    with pd.option_context('display.max_rows',None,'display.width',170):
        print(df.round(4).to_string(index=False),flush=True)
    print(f"ALL-PARAMETER GATE: {'PASS' if not fail.any() else 'FAIL/INCONCLUSIVE'} | failing {int(fail.sum())}/{int(okmask.sum())}",flush=True)
    print('FAILING PARAMETERS (worst first):',flush=True)
    if fail.any():print(df.loc[fail].sort_values(['ess_bulk','rhat']).round(4).to_string(index=False),flush=True)
    else:print(' none',flush=True)
    relevant=df.quantity.str.contains(':tail_|:s2_|log_collapsed_like|sigma2|hyper:|:bar_rho')
    relfail=fail & relevant
    print(f"ALLOCATION-RELEVANT GATE: {'PASS' if not relfail.any() else 'FAIL'} | failing {int(relfail.sum())}/{int(relevant.sum())}",flush=True)
    print('Median bulk ESS (descriptive, not a replacement for the minimum):',round(float(df.loc[okmask,'ess_bulk'].median()),1),flush=True)
    # Decision-specific stability is distinct from marginal chain convergence.
    # It is reported for diagnosis only and NEVER substitutes for the strict gate.
    print('ALLOCATION STABILITY ACROSS COLD CHAINS (descriptive; cannot override failed Rhat/ESS):',flush=True)
    try:
        for xi in (0.,1.):
            pooled=dp_allocate(posterior_scores(s,xi),c['budgets'])
            per_chain=[dp_allocate(posterior_scores(s[cc:cc+1],xi),c['budgets']) for cc in range(s.shape[0])]
            for B in c['budgets']:
                d=pooled[B]['d']
                choices=[per_chain[k][B]['d'] for k in range(s.shape[0])]
                print(f'  xi={xi:g} budget={B}: pooled={d}, agreement={sum(t==d for t in choices)}/{len(choices)}',flush=True)
    except Exception as exc:
        print('  Allocation stability unavailable:',repr(exc),flush=True)

    print(f"Core max Rhat={df.loc[okmask,'rhat'].max():.4f}, min bulk ESS={df.loc[okmask,'ess_bulk'].min():.1f}, min tail ESS={df.loc[okmask,'ess_tail'].min():.1f}",flush=True)
    if (df.loc[okmask,'rhat']>1.01).any():print('WARNING: stricter publication-standard Rhat<=1.01 not met for all core quantities.',flush=True)
    print('CHAIN-LEVEL DRIFT CHECKS (early retained mean -> late retained mean):',flush=True)
    for ci in range(len(chains)):
        mid=s.shape[1]//2
        if mid:
            lo=float(ll[ci,:mid].mean());hi=float(ll[ci,mid:].mean())
            x=float(np.mean(s[ci,:mid,0,:K[0]].sum(axis=-1)))
            y=float(np.mean(s[ci,mid:,0,:K[0]].sum(axis=-1)))
            print(f' chain={ci+1} logLik_mean={lo:.3f}->{hi:.3f}; predictor_1_total_s2={x:.5f}->{y:.5f}; kappa={kap[ci].mean():.3f}',flush=True)
    print('SLICE EFFORT: likelihood evaluations per update (not MH acceptance rates):',flush=True)
    effort=sum((p['effort'] for p in chains));updates=sum((p['updates'] for p in chains))
    print(pd.DataFrame({'predictor':np.arange(1,m+1),
                        'orient_U_calls':np.round(effort[:,0]/np.maximum(1,updates[:,0]),2),
                        'orient_V_calls':np.round(effort[:,1]/np.maximum(1,updates[:,1]),2),
                        'spectrum_calls':np.round(effort[:,2]/np.maximum(1,updates[:,2]),2)}).to_string(index=False),flush=True)
    he=sum((p['h_effort'] for p in chains));hn=sum((p['h_updates'] for p in chains))
    print('Hyper slice evaluations per update [bar0, bar_j, kappa]:',np.round(he/np.maximum(hn,1),2).tolist(),flush=True)
    print('IMPLIED EFFECTIVE DIMENSIONS (99% of spectral energy) — descriptive, NOT sampled exact ranks:',flush=True)
    for j in range(m):
        v=s[:,:,j,:K[j]].reshape(-1,K[j]);cumulative=np.cumsum(v,axis=-1)/np.maximum(1e-300,v.sum(-1,keepdims=True))
        ranks=1+np.argmax(cumulative>=.99,axis=1)
        print(f' j={j+1}: '+str({int(k):round(float(np.mean(ranks==k)),3) for k in range(1,K[j]+1)}),flush=True)
    print('='*80,flush=True)
    return (bool(not fail.any()),df,s)

# ---- Exact multiple-choice knapsack; predictor-specific posterior mean + xi*sd ----
def dp_allocate(scores,budgets):
    c=CFG;Bmax=max(budgets);m=c['m'];N=c['nlev'];K=c['klist']
    F=np.full((m+1,Bmax+1),np.inf);F[0,0]=0
    back=np.full((m+1,Bmax+1),-1,int)
    for j in range(m):
        for b in range(Bmax+1):
            if not np.isfinite(F[j,b]):continue
            for d in range(1,K[j]+1):
                b2=b+N[j]*d
                if b2<=Bmax and F[j,b]+scores[j][d-1]<F[j+1,b2]:
                    F[j+1,b2]=F[j,b]+scores[j][d-1];back[j+1,b2]=d
    out={}
    for B in budgets:
        b=int(np.argmin(F[m,:B+1]));ds=[0]*m;cost=b
        if not np.isfinite(F[m,b]):raise RuntimeError('No feasible allocation')
        for j in range(m,0,-1):
            d=int(back[j,b]);ds[j-1]=d;b-=N[j-1]*d
        out[B]=dict(d=tuple(ds),cost=cost,score=float(F[m,cost]))
    return out

def posterior_scores(s,xi):
    c=CFG;out=[]
    for j in range(c['m']):
        vals=s[:,:,j,:c['klist'][j]].reshape(-1,c['klist'][j])
        arr=[]
        for d in range(1,c['klist'][j]+1):
            tail=vals[:,d:].sum(1)
            arr.append(float(np.mean(tail)+xi*np.std(tail,ddof=1)))
        out.append(arr)
    return out

def score_discrepancy(mean_scores,oracle_scores):
    # Exact sup |sum_j (S_j(d_j)-L0_j(d_j))| over the Cartesian dimension set.
    # It does not require enumerating exponentially many allocations.
    deviations=[np.asarray(mean_scores[j])-np.asarray(oracle_scores[j]) for j in range(CFG['m'])]
    delta=max(abs(sum(float(x.max()) for x in deviations)),
              abs(sum(float(x.min()) for x in deviations)))
    base=sum(float(oracle_scores[j][0]) for j in range(CFG['m']))
    return (float(delta),float(delta/max(base,1e-12)))

def truth_scores(truth):
    return [[float(np.sum(np.asarray(truth['spec'][j][d:])**2)) for d in range(1,CFG['klist'][j]+1)] for j in range(CFG['m'])]

def print_posterior_recovery(s,truth):
    c=CFG
    print('PER-PREDICTOR POSTERIOR SPECTRUM vs GENERATING SPECTRUM:',flush=True)
    for j in range(c['m']):
        vals=s[:,:,j,:c['klist'][j]].reshape(-1,c['klist'][j])
        actual=np.asarray(truth['spec'][j])**2
        mean=vals.mean(0);lo,hi=np.quantile(vals,[.025,.975],axis=0)
        for k in range(c['klist'][j]):
            print(f'  predictor={j+1:02d} eigen={k+1} TRUE={actual[k]:.5f} POST_MEAN={mean[k]:.5f} CI95=({lo[k]:.5f},{hi[k]:.5f})',flush=True)
        for d in range(1,c['klist'][j]):
            v=vals[:,d:].sum(1);t=float(actual[d:].sum())
            print(f'  predictor={j+1:02d} tail(d={d}) TRUE={t:.5f} MEAN={np.mean(v):.5f} SD={np.std(v,ddof=1):.5f} CI95=({np.quantile(v,.025):.5f},{np.quantile(v,.975):.5f})',flush=True)

def phi_from_truth_data(truth,omega,dt):
    C,U,_=dt
    z=truth['mu']+U@truth['gamma']
    for j in range(CFG['m']):z+=truth['theta'][j][C[j]]
    return phi(z,omega)

# ---- Checkpoint-aware end-to-end replicate: no synthetic convergence claims ----
def run_one(task):
    global CFG
    sim,regime,ntrain,rep=task
    CFG=setup(sim)
    c=CFG
    name=f'Simulation{sim}_{regime}_n{ntrain}_rep{rep:03d}'
    key=str(task)
    STATE['task_name']=name
    w=STATE['work']
    if w.get('task')!=task:
        seed=(271828 if sim=='1' else 314159)+100003*rep+1009*c['scenarios'].index((regime if sim=='2' else f'{regime}_n{ntrain}',regime,ntrain))
        # In Simulation 1, the scenario's regime is clear/weak; in Simulation 2 the same as name.
        truth=target(regime)
        t=teacher(sim)
        w=dict(task=task,name=name,phase='reference',seed=seed,truth=truth,teacher=t,
               training=data(ntrain,truth,t,seed+1),validation=data(c['n_val'],truth,t,seed+2),
               test=data(c['n_test'],truth,t,seed+3))
        STATE['work']=w
        atomic_checkpoint(force=True)
    print(f'\n>>> {name} | phase={w["phase"]}',flush=True)
    if w['phase']=='reference':
        net,loss=fit_model(w['training'],'reference',w['seed']+10)
        w['ref_params']=reference_parameters(net)
        w['reference_train_mse']=loss
        w['reference_test_mse']=mse(w['test'][2],predict(net,w['test']))
        w['reference_validation_mse']=mse(w['validation'][2],predict(net,w['validation']))
        truth_train_mse=mse(w['training'][2],phi_from_truth_data(w['truth'],w['teacher'],w['training']))
        print(f'  REFERENCE ANCHOR: train={loss:.6f}, validation={w["reference_validation_mse"]:.6f}, '
              f'test={w["reference_test_mse"]:.6f}, true noise variance={c["noise_sd"]**2:.6f}, '
              f'generating train MSE={truth_train_mse:.6f}',flush=True)
        del net;gc.collect()
        w['phase']='mcmc';atomic_checkpoint(force=True)
    if w['phase']=='mcmc' and REFERENCE_PREFLIGHT_ONLY and 'ladders' not in w:
        print('REFERENCE PREFLIGHT COMPLETE (NO MCMC). Set REFERENCE_PREFLIGHT_ONLY=False; choose option 3.',flush=True)
        return dict(status='REFERENCE_PREFLIGHT',task=name)
    if w['phase']=='mcmc':
        configure_fast_likelihood(w['training'],w['ref_params']['omega'])
        if 'ladders' not in w:
            w['ladders']=[make_ladder(w['ref_params'],w['training'],w['seed']+10000+21111*i) for i in range(N_CHAINS)]
            w['target_iter']=MCMC_BASE
            atomic_checkpoint(force=True)
        ladders=w['ladders']
        while True:
            for i,L in enumerate(ladders):
                if L['it']<w['target_iter']:
                    print(f'  independent PT ladder {i+1}/{N_CHAINS}: {L["it"]} -> {w["target_iter"]}',flush=True)
                while L['it']<w['target_iter']:
                    pt_one_sweep(L,w['training'],w['ref_params']['omega'])
                    if L['it']%100==0:
                        if L['it']%500==0:
                            print(f'   PT ladder {i+1} iter={L["it"]} cold_logLik={L["replicas"][0]["ll"]:.2f} '
                                  f'roundtrips={L["roundtrips"]}',flush=True)
                        atomic_checkpoint()
                atomic_checkpoint(force=True)
            chains=[L['replicas'][0] for L in ladders]
            pt_report(ladders)
            passed,diagnostics_df,s=diagnostics(chains,name,w['target_iter'])
            if passed:
                w['passed']=True;break
            if w['target_iter']>=MCMC_MAX:
                w['passed']=False;break
            w['target_iter']=min(MCMC_MAX,w['target_iter']+MCMC_EXTEND)
            print('EXTENDING PARALLEL-TEMPERING LADDERS for failed/inconclusive diagnostics to',w['target_iter'],flush=True)
            atomic_checkpoint(force=True)
        w['posterior_s2']=s
        w['posterior_summary']=diagnostics_df.to_dict('records')
        # Save diagnostic metadata, including failures, independently of console output.
        path_csv=os.path.join(RUN_DIR,name+'_mcmc_diagnostics.csv')
        tmp_csv=path_csv+'.tmp'
        diagnostics_df.to_csv(tmp_csv,index=False)
        os.replace(tmp_csv,path_csv)
        if not w['passed']:
            print('BAYESIAN CONVERGENCE FAILED/INCONCLUSIVE. Detailed printed diagnostics above.',flush=True)
            if REQUIRE_CONVERGENCE:
                w['phase']='failed';atomic_checkpoint(force=True)
                return dict(status='MCMC_FAILED',task=name,iterations=w['target_iter'])
        # retain only posterior draws; no large MCMC state needed after accepted checks
        h0=np.stack([p['traces']['bar0'] for p in chains]);bar=np.stack([p['traces']['bar'] for p in chains]);kap=np.stack([p['traces']['kappa'] for p in chains])
        print('HIERARCHICAL POSTERIOR MEANS: hypermean=',float(h0.mean()),'kappa=',float(kap.mean()),flush=True)
        print('per predictor mean ratios=',np.round(bar.mean((0,1)),4),flush=True)
        print_posterior_recovery(s,w['truth'])
        # Persist posterior summaries after convergence; these files permit
        # reproduction of spectra/figures without rerunning hours of MCMC.
        dest=os.path.join(RUN_DIR,name+'_converged_posterior.npz')
        tmp=dest+'.tmp'
        with open(tmp,'wb') as f:
            np.savez_compressed(f,squared_singular_values=s,bar_rho=bar,
                                bar_rho0=h0,kappa=kap,
                                log_likelihood=np.stack([p['traces']['ll'] for p in chains]),
                                schema=np.array('Simulation2_V7_fixed_rank_hierarchical_beta'))
            f.flush();os.fsync(f.fileno())
        os.replace(tmp,dest)
        print('  POSTERIOR SAMPLES SAVED:',dest,flush=True)
        w.pop('ladders',None)
        alloc={};tgt=dp_allocate(truth_scores(w['truth']),c['budgets'])
        delta,relative_delta=score_discrepancy(posterior_scores(s,0),truth_scores(w['truth']))
        w['spectral_delta']=delta;w['normalized_spectral_delta']=relative_delta
        squared_errors=[];tail_errors=[];tail_coverage=[]
        for j in range(c['m']):
            vals=s[:,:,j,:c['klist'][j]].reshape(-1,c['klist'][j]);actual=np.asarray(w['truth']['spec'][j])**2
            squared_errors.extend((vals.mean(0)-actual)**2)
            for d in range(1,c['klist'][j]):
                tt=vals[:,d:].sum(1)
                tail_errors.append((tt.mean()-actual[d:].sum())**2)
                if actual[d:].sum()>0:
                    tail_coverage.append(float(np.quantile(tt,.025)<=actual[d:].sum()<=np.quantile(tt,.975)))
        w['posterior_s2_rmse']=float(np.sqrt(np.mean(squared_errors)))
        w['posterior_tail_rmse']=float(np.sqrt(np.mean(tail_errors)))
        w['posterior_tail_coverage']=float(np.mean(tail_coverage)) if tail_coverage else float('nan')
        w['nonzero_tails_for_inclusion']=len(tail_coverage)
        print('POSTERIOR RECOVERY: s2 RMSE=',w['posterior_s2_rmse'],
              'tail RMSE=',w['posterior_tail_rmse'], '95% inclusion, positive true tails only=',w['posterior_tail_coverage'],
              'number of positive true tails=',len(tail_coverage),
              'score sup discrepancy=',delta,'normalized=',relative_delta,flush=True)
        for xi in RISK_XI:
            aa=dp_allocate(posterior_scores(s,xi),c['budgets'])
            alloc[xi]=aa
            print(f'ALLOCATIONS xi={xi}:',flush=True)
            for B in c['budgets']:
                print(' budget=',B,' posterior=',aa[B]['d'],'score=',round(aa[B]['score'],5),
                      'true spectral oracle=',tgt[B]['d'],flush=True)
        w['alloc']=alloc;w['oracle']=tgt
        w['refits']={};w['phase']='refit';atomic_checkpoint(force=True)
    if w['phase']=='failed':return dict(status='MCMC_FAILED',task=name,iterations=w.get('target_iter'))
    if w['phase']=='refit':
        wanted=sorted(set(tuple(v['d']) for aa in w['alloc'].values() for v in aa.values()))
        print('REFITTING',len(wanted),'UNIQUE ALLOCATIONS across risk coefficients/budgets',flush=True)
        for d in wanted:
            if d in w['refits']:continue
            net,fit_info=fit_reduced_v7(w['ref_params'],w['training'],w['validation'],d,
                                         w['seed']+2000+sum((i+1)*z for i,z in enumerate(d)))
            v=fit_info['val']
            te=mse(w['test'][2],predict(net,w['test']))
            w['refits'][d]=dict(train=fit_info['train'],val=v,test=te,epochs=fit_info['epochs'],method=fit_info['method'])
            print('  REFIT',d,'cost=',sum(c['nlev'][j]*d[j] for j in range(c['m'])),
                  'VAL=',round(v,6),'TEST=',round(te,6),'epochs=',fit_info['epochs'],
                  'method=',fit_info['method'],flush=True)
            del net;gc.collect();atomic_checkpoint(force=True)
        w['phase']='summary';atomic_checkpoint(force=True)
    if w['phase']=='summary':
        results=[];baseline=max(c['budgets'])
        for xi,aa in w['alloc'].items():
            largest=w['refits'][aa[baseline]['d']]['val']
            eligible=[B for B in c['budgets'] if w['refits'][aa[B]['d']]['val']<=largest+config['zeta']]
            Bsel=min(eligible)
            dsel=aa[Bsel]['d'];best=w['refits'][dsel]
            exact=np.mean([aa[B]['d']==w['oracle'][B]['d'] for B in c['budgets']])
            true_scores=truth_scores(w['truth'])
            mean_true_regret=np.mean([sum(true_scores[j][aa[B]['d'][j]-1] for j in range(c['m']))-
                                    w['oracle'][B]['score'] for B in c['budgets']])
            oracle_tie_aware=np.mean([abs(sum(true_scores[j][aa[B]['d'][j]-1] for j in range(c['m']))-
                                          w['oracle'][B]['score'])<1e-9 for B in c['budgets']])
            test_oracle_B=min(c['budgets'],key=lambda B:w['refits'][aa[B]['d']]['test'])
            out=dict(status='OK' if w['passed'] else 'MCMC_FAILED_WARNING',task=name,sim=sim,regime=regime,ntrain=ntrain,rep=rep,
                     xi=float(xi),Bsel=Bsel,dsel=str(dsel),cost=int(sum(c['nlev'][j]*dsel[j] for j in range(c['m']))),
                     val_mse=best['val'],test_mse=best['test'],baseline_max_test_mse=w['refits'][aa[baseline]['d']]['test'],
                     ref_test_mse=w['reference_test_mse'],oracle_budget_allocation_exact_rate=exact,
                     oracle_budget_allocation_tie_aware_rate=oracle_tie_aware,
                     mean_true_spectral_regret=mean_true_regret,
                     n_nonzero_tails_for_interval_inclusion=w['nonzero_tails_for_inclusion'],
                     test_oracle_B=test_oracle_B, budget_match_with_test_oracle=int(Bsel==test_oracle_B),
                     test_oracle_mse=w['refits'][aa[test_oracle_B]['d']]['test'],
                     test_excess_vs_max_budget=best['test']-w['refits'][aa[baseline]['d']]['test'],
                     s2_rmse=w['posterior_s2_rmse'],tail_rmse=w['posterior_tail_rmse'],
                     tail_coverage=w['posterior_tail_coverage'],spectral_delta=w['spectral_delta'],
                     normalized_spectral_delta=w['normalized_spectral_delta'],
                     target_dim_accuracy=float(np.mean(np.asarray(dsel)==w['truth']['dtrue'])) if w['truth']['dtrue'] else np.nan,
                     mcmc_iters=int(w['target_iter']))
            results.append(out)
            print('\nSELECTED RESULT:',out,flush=True)
        w['results']=results
        w['phase']='done';atomic_checkpoint(force=True)
        return results
    if w['phase']=='done':return w['results']
    raise RuntimeError('Unexpected phase: '+str(w['phase']))

TASKS=[]
for sim in SIMULATIONS:
    c=setup(sim)
    for label,regime,n in c['scenarios']:
        for rep in range(1,c['num_reps']+1):TASKS.append((sim,regime,n,rep))
print('\nCONFIG:',json.dumps(config,indent=2),'\nTOTAL REPLICATION TASKS:',len(TASKS),'\n',flush=True)
try:
    for itask,task in enumerate(TASKS,1):
        k=str(task)
        if k in STATE['completed']:continue
        print(f'\n[{itask}/{len(TASKS)}] START TASK {task}',flush=True)
        res=run_one(task)
        if isinstance(res,dict) and res.get('status')=='REFERENCE_PREFLIGHT':
            atomic_checkpoint(force=True)
            print('PREFLIGHT STOP: checkpoint preserved; no task marked complete.',flush=True)
            break
        STATE['completed'][k]=res if isinstance(res,list) else [res]
        # Preserve the failing chains, traces, and reference fit rather than erasing
        # every diagnostic draw when a convergence failure occurs.
        if not any(r.get('status')=='MCMC_FAILED' for r in STATE['completed'][k]):
            STATE['work']={}
        atomic_checkpoint(force=True)
        if STOP_ON_FIRST_FAILURE and any(r.get('status')=='MCMC_FAILED' for r in STATE['completed'][k]):
            print('STOPPED AFTER NONCONVERGENCE: inspect the printed diagnostics before rerunning or changing the model.',flush=True)
            break
except KeyboardInterrupt:
    print('\nUSER INTERRUPTION: saving last known valid checkpoint.',flush=True)
    atomic_checkpoint(force=True);raise
except Exception:
    print('\nERROR DETECTED (not suppressed). Last state checkpoint follows:',flush=True)
    traceback.print_exc();atomic_checkpoint(force=True);raise

ALL=[x for v in STATE['completed'].values() for x in v]
RESULT_DF=pd.DataFrame(ALL)
print('\n\nALL REPLICATION RESULTS\n',RESULT_DF.to_string(index=False),flush=True)
if not RESULT_DF.empty and 'status' in RESULT_DF and 'xi' in RESULT_DF:
    ok=RESULT_DF[RESULT_DF['status']=='OK']
    if len(ok):
        print('\nGROUPED SIMULATION RESULTS\n',ok.groupby(['sim','regime','ntrain','xi'],dropna=False)
            .agg(n=('rep','count'),mean_selected_B=('Bsel','mean'),mean_test_mse=('test_mse','mean'),
                 mean_reference_test_mse=('ref_test_mse','mean'),mean_allocation_exact=('oracle_budget_allocation_exact_rate','mean'),
                 mean_target_dim_accuracy=('target_dim_accuracy','mean'),
                 mean_tail_rmse=('tail_rmse','mean'),mean_spectral_delta=('spectral_delta','mean'),
                 mean_normalized_delta=('normalized_spectral_delta','mean'),
                 budget_test_oracle_match=('budget_match_with_test_oracle','mean'),
                 mean_test_excess=('test_excess_vs_max_budget','mean')).round(5).to_string(),flush=True)
if not RESULT_DF.empty:
    output_csv=os.path.join(RUN_DIR,'main_results.csv')
    tmp_csv=output_csv+'.tmp'
    RESULT_DF.to_csv(tmp_csv,index=False)
    os.replace(tmp_csv,output_csv)
    print('MAIN RESULTS SAVED:',output_csv,flush=True)
print('\nCOMPLETED; all MCMC diagnostics printed; final diagnostic CSV and posterior saved on PASS. Checkpoint:',CPATH,flush=True)
print('A PASS is evidence from these diagnostics, not proof of exact posterior convergence. No rank probability is estimated.',flush=True)
print('TRUE-ZERO TAILS: 95% intervals from a strictly positive spectral prior cannot include exact zero;',
      'the inclusion rate excludes zero true tails and is not a frequentist coverage estimate.',flush=True)


NumPy BLAS threads limited to 1 for small-matrix MCMC computations.
Torch 2.11.0+cpu | device: cpu | sampler: cached likelihood/auto JIT + conditional hyperprior; neural nets: torch GPU if available
Mounted at /content/drive

RUN SELECTION | /content/drive/MyDrive/JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V7_EXACT_TAIL_MIXING/Simulation2/diagnostic
[1] START FRESH V7 (new data, calibrated reference, fresh posterior)
[2] REUSE V6/V5/V4 DATA ONLY, RETRAIN calibrated reference, START new V7 posterior
[3] RESUME V7 CHECKPOINT (including after reference-only preflight)
Available data imports (old checkpoints read-only): [('V6', '/content/drive/MyDrive/JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V6_TAIL_MH/Simulation2/diagnostic/checkpoint.pkl'), ('V4', '/content/drive/MyDrive/JRSSB_Embedding_Allocation/FIXED_RANK_HIERARCHICAL_SHRINKAGE_PT_SIM2_V4_FUSED/Simulation2/diagnostic/checkpoint.pkl')]
REUSED V6 DATA FROM /content/drive/MyDrive/JRSS